# Computer Vision — Assignment 3  
## From Eigenspace to Gaussian Image Space: 2D Gaussian Splatting and PCA-based Initialization

This notebook implements the full assignment pipeline:

1. A fully differentiable 2D Gaussian renderer.
2. Single-image optimization with a fixed number of Gaussians.
3. Adaptive density control with pruning, cloning, and splitting.
4. PCA/eigenspace analysis and an MLP that predicts Gaussian parameters.
5. A controlled benchmark comparing random initialization with PCA-based initialization.

The notebook is designed to be run from top to bottom in Google Colab.  
All project paths are relative to the notebook directory.

> Recommended workflow: start with `RUN_MODE = "debug"`, verify that every section runs, then switch to `medium`, and only then run `full`.

### Two-stage Colab execution

The notebook supports a CPU preparation stage followed by a GPU pipeline stage. Persistent artifacts are stored in Google Drive so that changing the Colab accelerator does not require downloading or preprocessing the datasets again.


## 0. Engineering principles

The implementation follows several rules intended to reduce runtime and avoid repeated work:

- One renderer is reused throughout all five parts.
- Dataset subsets and their indices are deterministic and cached.
- Dataset cache misses use Hugging Face first, with torchvision as a fallback.
- Raw downloads stay on the local Colab filesystem; prepared caches stay persistent.
- PCA is fitted once per dataset and reused.
- Predictor checkpoints and experiment results are saved and reloaded.
- Optimization histories are collected during the same run that produces the reconstruction.
- Pixel grids are cached in memory.
- The renderer is vectorized over pixels and Gaussians.
- A chunked renderer is available if the full renderer causes an out-of-memory error.
- Every expensive experiment has a cache key that depends on its configuration.

In [ ]:
# ============================================================
# 0. Runtime configuration
# ============================================================

# Use the SAME RUN_MODE during the CPU preparation run and the later GPU run.
# Example:
#   CPU session: RUN_MODE="full", EXECUTION_STAGE="prepare"
#   GPU session: RUN_MODE="full", EXECUTION_STAGE="pipeline"
RUN_MODE = "full"       # "debug", "medium", or "full"
EXECUTION_STAGE = "pipeline"   # "prepare" or "pipeline"

USE_CACHE = True
FORCE_RECOMPUTE = False
SEED = 42

# Persistent storage is required if the Colab runtime is changed from CPU to GPU.
# Colab may allocate a new VM, so files under the temporary /content filesystem
# cannot be assumed to survive. The notebook mounts Google Drive and then changes
# the working directory to a project workspace. All project paths below remain
# relative to that workspace.
PERSIST_TO_GOOGLE_DRIVE = True
DRIVE_WORKSPACE_NAME = "CV_Assignment_3_workspace"

assert EXECUTION_STAGE in {"prepare", "pipeline"}

ENABLE_PART_2 = EXECUTION_STAGE == "pipeline"
ENABLE_PART_3 = EXECUTION_STAGE == "pipeline"
ENABLE_PCA = True
ENABLE_PREDICTOR = EXECUTION_STAGE == "pipeline"
ENABLE_PART_5 = EXECUTION_STAGE == "pipeline"

RUN_PROFILES = {
    "debug": {
        "train_cap": {
            "MNIST": 40,
            "FashionMNIST": 40,
            "CIFAR10": 40,
            "Olivetti": 40,
            "STL10": 20,
        },
        "test_cap": {
            "MNIST": 10,
            "FashionMNIST": 10,
            "CIFAR10": 10,
            "Olivetti": 10,
            "STL10": 6,
        },
        "pca_cap": {
            "MNIST": 40,
            "FashionMNIST": 40,
            "CIFAR10": 40,
            "Olivetti": 40,
            "STL10": 20,
        },
        "pca_k_cap": 24,
        "predictor_epochs": 1,
        "predictor_batch_size": 4,
        "part2_iterations": 40,
        "part2_capacity_values": [50],
        "part3_iterations": 80,
        "part3_interval": 20,
        "part3_max_gaussians": 80,
        "benchmark_iterations": 40,
        "benchmark_images": 2,
        "snapshot_count": 4,
    },
    "medium": {
        "train_cap": {
            "MNIST": 3000,
            "FashionMNIST": 3000,
            "CIFAR10": 3000,
            "Olivetti": 280,
            "STL10": 1200,
        },
        "test_cap": {
            "MNIST": 100,
            "FashionMNIST": 100,
            "CIFAR10": 100,
            "Olivetti": 80,
            "STL10": 80,
        },
        "pca_cap": {
            "MNIST": 3000,
            "FashionMNIST": 3000,
            "CIFAR10": 3000,
            "Olivetti": 280,
            "STL10": 1200,
        },
        "pca_k_cap": 128,
        "predictor_epochs": 8,
        "predictor_batch_size": 8,
        "part2_iterations": 350,
        "part2_capacity_values": [50, 100, 200, 400],
        "part3_iterations": 700,
        "part3_interval": 50,
        "part3_max_gaussians": 400,
        "benchmark_iterations": 350,
        "benchmark_images": 10,
        "snapshot_count": 7,
    },
    "full": {
        "train_cap": {
            "MNIST": 10000,
            "FashionMNIST": 10000,
            "CIFAR10": 10000,
            "Olivetti": 320,
            "STL10": 3000,
        },
        "test_cap": {
            "MNIST": 250,
            "FashionMNIST": 250,
            "CIFAR10": 250,
            "Olivetti": 80,
            "STL10": 200,
        },
        "pca_cap": {
            "MNIST": 10000,
            "FashionMNIST": 10000,
            "CIFAR10": 10000,
            "Olivetti": 320,
            "STL10": 3000,
        },
        "pca_k_cap": 256,
        "predictor_epochs": 25,
        "predictor_batch_size": 8,
        "part2_iterations": 1000,
        "part2_capacity_values": [50, 100, 200, 400],
        "part3_iterations": 1600,
        "part3_interval": 80,
        "part3_max_gaussians": 1200,
        "benchmark_iterations": 1000,
        "benchmark_images": 12,
        "snapshot_count": 9,
    },
}

assert RUN_MODE in RUN_PROFILES
CFG = RUN_PROFILES[RUN_MODE]
print("RUN_MODE =", RUN_MODE)
print("EXECUTION_STAGE =", EXECUTION_STAGE)
print(CFG)

## CPU preparation run and GPU execution run

Colab can allocate a different virtual machine when the accelerator changes. Therefore, a cache stored only in the temporary runtime filesystem may disappear when switching from CPU to GPU.

This notebook stores its workspace in Google Drive when `PERSIST_TO_GOOGLE_DRIVE = True`.

### First run — CPU

Use:

```python
RUN_MODE = "full"            # or the mode intended for the later GPU run
EXECUTION_STAGE = "prepare"
```

Run all cells. This stage:

- downloads each dataset only if it is not already present;
- creates deterministic preprocessed dataset caches;
- computes and caches PCA artifacts;
- skips Gaussian optimization, adaptive density control, predictor training, and the benchmark.

### Second run — GPU

Change only:

```python
EXECUTION_STAGE = "pipeline"
```

Keep the same `RUN_MODE`. After selecting a GPU runtime, run all cells again. Dataset and PCA cells should print `cache hit`, while GPU-dependent training and optimization continue normally.

The same `RUN_MODE` must be used in both sessions because subset sizes and cache signatures depend on it.

In [ ]:
# ============================================================
# 1. Imports, persistent paths, device, and reproducibility
# ============================================================

import os
import gc
import json
import math
import time
import random
import hashlib
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

from torchvision import datasets as torchvision_datasets, transforms

try:
    from datasets import load_dataset as hf_load_dataset
    HF_DATASETS_AVAILABLE = True
except ImportError:
    # Colab images do not always ship with Hugging Face Datasets. Install it
    # only when missing; local environments can still use the torchvision fallback.
    try:
        import subprocess
        import sys
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "datasets"]
        )
        from datasets import load_dataset as hf_load_dataset
        HF_DATASETS_AVAILABLE = True
    except Exception as hf_import_error:
        hf_load_dataset = None
        HF_DATASETS_AVAILABLE = False
        print(
            "[dataset backend warning] Hugging Face Datasets is unavailable; "
            f"torchvision fallback will be used. Reason: {hf_import_error}"
        )
from sklearn.datasets import fetch_olivetti_faces
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Persistent Colab workspace
# ------------------------------------------------------------
# The Google Drive mount point is a Colab system path. Project files themselves
# still use relative paths after changing into the persistent workspace.

IN_COLAB = False
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB and PERSIST_TO_GOOGLE_DRIVE:
    from google.colab import drive  # type: ignore

    # Use Colab's standard mount point. The previous implementation derived the
    # mount location from Path.cwd(), which can point to a temporary directory
    # that no longer exists after a runtime restart.
    drive_mount = Path("/content/drive")
    drive_mount.parent.mkdir(parents=True, exist_ok=True)

    drive.mount(str(drive_mount), force_remount=False)

    workspace = drive_mount / "MyDrive" / DRIVE_WORKSPACE_NAME
    workspace.mkdir(parents=True, exist_ok=True)
    os.chdir(workspace)
    print("Persistent workspace:", Path.cwd())
else:
    print("Local workspace:", Path.cwd())

PROJECT_ROOT = Path(".")

# Prepared tensors and experiment artifacts remain persistent. Raw downloads are
# temporary and local in Colab because many small writes to Google Drive are slow.
if IN_COLAB:
    DATA_DIR = Path("/content/cv_assignment_3_raw_data")
else:
    DATA_DIR = PROJECT_ROOT / "data"

CACHE_DIR = PROJECT_ROOT / "cache"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"

for p in [DATA_DIR, CACHE_DIR, CHECKPOINT_DIR, RESULTS_DIR, FIGURES_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(SEED)

print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Data directory:", DATA_DIR.resolve())
print("Cache directory:", CACHE_DIR.resolve())

## 1. Mathematical derivation of the inverse covariance

For each Gaussian,

$
\Sigma = R(\theta)
\begin{bmatrix}
s_x^2 & 0\\
0 & s_y^2
\end{bmatrix}
R(\theta)^\top
$

where

$
R(\theta)=
\begin{bmatrix}
\cos\theta & -\sin\theta\\
\sin\theta & \cos\theta
\end{bmatrix}.
$

Because $R$ is orthogonal,

$
\Sigma^{-1}
=
R(\theta)
\begin{bmatrix}
1/s_x^2 & 0\\
0 & 1/s_y^2
\end{bmatrix}
R(\theta)^\top.
$

Let $u=\cos\theta$ and $v=\sin\theta$. Then

$
\Sigma^{-1}=
\begin{bmatrix}
a & b\\
b & c
\end{bmatrix}
$

with

$
a=\frac{u^2}{s_x^2}+\frac{v^2}{s_y^2},
\qquad
b=uv\left(\frac{1}{s_x^2}-\frac{1}{s_y^2}\right),
\qquad
c=\frac{v^2}{s_x^2}+\frac{u^2}{s_y^2}.
$

For a pixel $p=(x,y)$ and Gaussian center $\mu=(\mu_x,\mu_y)$, define

$
d_x=x-\mu_x,\qquad d_y=y-\mu_y.
$

The scalar Mahalanobis form is

$
(p-\mu)^\top\Sigma^{-1}(p-\mu)
=
a d_x^2 + 2b d_xd_y + c d_y^2.
$

Therefore,

$
G(p)=\exp\left(
-\frac12
\left[
a d_x^2 + 2b d_xd_y + c d_y^2
\right]
\right).
$

This direct formula avoids numerical matrix inversion and is fully differentiable.

In [ ]:
# ============================================================
# 2. Shared utilities and data structures
# ============================================================

def config_hash(payload: Any) -> str:
    text = json.dumps(payload, sort_keys=True, default=str)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:12]

def inverse_sigmoid(x: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    x = x.clamp(eps, 1.0 - eps)
    return torch.log(x / (1.0 - x))

def inverse_softplus(x: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    x = x.clamp_min(eps)
    return torch.log(torch.expm1(x))

def mse_to_psnr(mse: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    return -10.0 * torch.log10(mse.clamp_min(eps))

@dataclass
class OptimizationResult:
    final_raw_gaussians: torch.Tensor
    final_image: torch.Tensor
    loss_history: List[float]
    psnr_history: List[float]
    gaussian_count_history: List[int]
    snapshots: Dict[int, Dict[str, torch.Tensor]]
    runtime_seconds: float

@dataclass
class DatasetBundle:
    name: str
    train_images: torch.Tensor
    test_images: torch.Tensor
    image_shape: Tuple[int, int, int]
    train_indices: torch.Tensor
    test_indices: torch.Tensor

def save_torch(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(obj, path)

def load_torch(path: Path) -> Any:
    return torch.load(path, map_location="cpu", weights_only=False)

def make_snapshot_iterations(total_iterations: int, count: int) -> List[int]:
    if total_iterations <= 1:
        return [0]
    points = np.unique(
        np.round(np.geomspace(1, total_iterations, num=max(count - 1, 2))).astype(int)
    )
    points = [0] + [int(x) for x in points if x < total_iterations] + [total_iterations - 1]
    return sorted(set(points))

## 2. Parameterization of the Gaussian primitives

The learnable tensor has shape \((N,9)\) or \((B,N,9)\), with the following order:

1. raw $x$
2. raw $y$
3. raw $s_x$
4. raw $s_y$
5. $\theta$
6. raw red
7. raw green
8. raw blue
9. raw opacity

The raw values are converted to valid values as follows:

- position: sigmoid $\rightarrow [0,1]$
- scale: softplus plus a positive minimum
- angle: left unrestricted
- color: sigmoid $\rightarrow [0,1]$
- opacity: sigmoid $\rightarrow [0,1]$

Using smooth activations keeps the complete renderer differentiable.

In [ ]:
# ============================================================
# 3. Gaussian parameter utilities
# ============================================================

MIN_SCALE = 1e-3
INITIAL_SCALE = 0.08

def activate_gaussians(
    raw_gaussians: torch.Tensor,
    min_scale: float = MIN_SCALE,
) -> Dict[str, torch.Tensor]:
    if raw_gaussians.ndim == 2:
        raw_gaussians = raw_gaussians.unsqueeze(0)
    if raw_gaussians.ndim != 3 or raw_gaussians.shape[-1] != 9:
        raise ValueError("Expected shape (N,9) or (B,N,9).")

    positions = torch.sigmoid(raw_gaussians[..., 0:2])
    scales = F.softplus(raw_gaussians[..., 2:4]) + min_scale
    angles = raw_gaussians[..., 4]
    colors = torch.sigmoid(raw_gaussians[..., 5:8])
    opacities = torch.sigmoid(raw_gaussians[..., 8])

    return {
        "positions": positions,
        "scales": scales,
        "angles": angles,
        "colors": colors,
        "opacities": opacities,
    }

def initialize_random_gaussians(
    num_gaussians: int,
    batch_size: int = 1,
    initial_scale: float = INITIAL_SCALE,
    initial_opacity: float = 0.35,
    device: torch.device = DEVICE,
    seed: Optional[int] = None,
) -> torch.Tensor:
    gen = torch.Generator(device="cpu")
    gen.manual_seed(SEED if seed is None else seed)

    positions = torch.rand(batch_size, num_gaussians, 2, generator=gen)
    scales = torch.full((batch_size, num_gaussians, 2), initial_scale)
    angles = (torch.rand(batch_size, num_gaussians, generator=gen) - 0.5) * math.pi
    colors = torch.rand(batch_size, num_gaussians, 3, generator=gen)
    opacities = torch.full((batch_size, num_gaussians, 1), initial_opacity)

    raw = torch.cat(
        [
            inverse_sigmoid(positions),
            inverse_softplus((scales - MIN_SCALE).clamp_min(1e-5)),
            angles.unsqueeze(-1),
            inverse_sigmoid(colors),
            inverse_sigmoid(opacities),
        ],
        dim=-1,
    ).to(device=device, dtype=DTYPE)

    return raw[0] if batch_size == 1 else raw

## 3. Differentiable alpha compositing

For each Gaussian,

$
w_i(p)=\alpha_i G_i(p).
$

The transmittance before Gaussian $i$ is

$
T_i(p)=\prod_{j<i}(1-w_j(p)).
$

The rendered color is

$
C(p)=\sum_i c_i w_i(p)T_i(p)+T_{N+1}(p)c_{\text{bg}}.
$

The renderer below performs this operation with an exclusive cumulative product.  
Index 0 is treated as the frontmost Gaussian.

In [ ]:
# ============================================================
# 4. Pixel-grid cache and differentiable renderer
# ============================================================

_PIXEL_GRID_CACHE: Dict[Tuple[int, int, str, torch.dtype], Tuple[torch.Tensor, torch.Tensor]] = {}

def get_pixel_grid(
    height: int,
    width: int,
    device: torch.device,
    dtype: torch.dtype,
) -> Tuple[torch.Tensor, torch.Tensor]:
    key = (height, width, str(device), dtype)
    if key not in _PIXEL_GRID_CACHE:
        ys = torch.linspace(0.0, 1.0, height, device=device, dtype=dtype)
        xs = torch.linspace(0.0, 1.0, width, device=device, dtype=dtype)
        grid_y, grid_x = torch.meshgrid(ys, xs, indexing="ij")
        _PIXEL_GRID_CACHE[key] = (
            grid_x.view(1, 1, height, width),
            grid_y.view(1, 1, height, width),
        )
    return _PIXEL_GRID_CACHE[key]

class GaussianRenderer2D(nn.Module):
    def __init__(
        self,
        height: int,
        width: int,
        background_color: Tuple[float, float, float] = (0.0, 0.0, 0.0),
        min_scale: float = MIN_SCALE,
        chunk_size: Optional[int] = None,
    ):
        super().__init__()
        self.height = height
        self.width = width
        self.min_scale = min_scale
        self.chunk_size = chunk_size
        self.register_buffer(
            "background_color",
            torch.tensor(background_color, dtype=DTYPE).view(1, 3, 1, 1),
        )

    def _compute_kernel_and_weights(
        self,
        params: Dict[str, torch.Tensor],
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        positions = params["positions"]
        scales = params["scales"]
        angles = params["angles"]
        opacities = params["opacities"]

        bsz, _, _ = positions.shape
        grid_x, grid_y = get_pixel_grid(
            self.height, self.width, positions.device, positions.dtype
        )

        mu_x = positions[..., 0].view(bsz, -1, 1, 1)
        mu_y = positions[..., 1].view(bsz, -1, 1, 1)
        sx = scales[..., 0].view(bsz, -1, 1, 1)
        sy = scales[..., 1].view(bsz, -1, 1, 1)
        theta = angles.view(bsz, -1, 1, 1)

        u = torch.cos(theta)
        v = torch.sin(theta)

        inv_sx2 = 1.0 / (sx * sx)
        inv_sy2 = 1.0 / (sy * sy)

        a = u * u * inv_sx2 + v * v * inv_sy2
        b = u * v * (inv_sx2 - inv_sy2)
        c = v * v * inv_sx2 + u * u * inv_sy2

        dx = grid_x - mu_x
        dy = grid_y - mu_y

        quadratic = a * dx.square() + 2.0 * b * dx * dy + c * dy.square()
        kernel = torch.exp(-0.5 * quadratic)
        weights = opacities.view(bsz, -1, 1, 1) * kernel
        return kernel, weights

    @staticmethod
    def _composite_full(
        colors: torch.Tensor,
        weights: torch.Tensor,
        background: torch.Tensor,
    ) -> torch.Tensor:
        bsz, num_gaussians, height, width = weights.shape
        one_minus = 1.0 - weights

        prefix = torch.cat(
            [
                torch.ones(
                    bsz, 1, height, width,
                    device=weights.device,
                    dtype=weights.dtype,
                ),
                one_minus,
            ],
            dim=1,
        )
        trans_full = torch.cumprod(prefix, dim=1)
        trans_before = trans_full[:, :-1]
        final_trans = trans_full[:, -1]

        contribution = weights * trans_before
        image = (
            colors.view(bsz, num_gaussians, 3, 1, 1)
            * contribution.unsqueeze(2)
        ).sum(dim=1)
        image = image + background * final_trans.unsqueeze(1)
        return image

    @staticmethod
    def _composite_chunked(
        colors: torch.Tensor,
        weights: torch.Tensor,
        background: torch.Tensor,
        chunk_size: int,
    ) -> torch.Tensor:
        bsz, num_gaussians, height, width = weights.shape
        accumulated = torch.zeros(
            bsz, 3, height, width,
            device=weights.device,
            dtype=weights.dtype,
        )
        incoming_trans = torch.ones(
            bsz, height, width,
            device=weights.device,
            dtype=weights.dtype,
        )

        for start in range(0, num_gaussians, chunk_size):
            end = min(start + chunk_size, num_gaussians)
            w = weights[:, start:end]
            c = colors[:, start:end]

            one_minus = 1.0 - w
            prefix = torch.cat(
                [
                    torch.ones(
                        bsz, 1, height, width,
                        device=w.device,
                        dtype=w.dtype,
                    ),
                    one_minus,
                ],
                dim=1,
            )
            local_trans_full = torch.cumprod(prefix, dim=1)
            local_trans_before = local_trans_full[:, :-1]
            local_final_trans = local_trans_full[:, -1]

            effective = incoming_trans.unsqueeze(1) * local_trans_before * w
            accumulated = accumulated + (
                c.view(bsz, end - start, 3, 1, 1)
                * effective.unsqueeze(2)
            ).sum(dim=1)

            incoming_trans = incoming_trans * local_final_trans

        return accumulated + background * incoming_trans.unsqueeze(1)

    def forward(
        self,
        raw_gaussians: torch.Tensor,
        return_aux: bool = False,
    ):
        single_input = raw_gaussians.ndim == 2
        if single_input:
            raw_gaussians = raw_gaussians.unsqueeze(0)

        params = activate_gaussians(raw_gaussians, self.min_scale)
        kernel, weights = self._compute_kernel_and_weights(params)

        background = self.background_color.to(
            device=raw_gaussians.device,
            dtype=raw_gaussians.dtype,
        )

        if self.chunk_size is None:
            image = self._composite_full(params["colors"], weights, background)
        else:
            image = self._composite_chunked(
                params["colors"], weights, background, self.chunk_size
            )

        if single_input:
            image = image[0]
            if return_aux:
                aux = {k: v[0] for k, v in params.items()}
                aux["kernel"] = kernel[0]
                aux["weights"] = weights[0]
                return image, aux
            return image

        if return_aux:
            params["kernel"] = kernel
            params["weights"] = weights
            return image, params
        return image

## 4. Renderer correctness tests

The assignment requires the following checks:

- a single Gaussian peaks at its center;
- an isotropic Gaussian is rotationally symmetric;
- a larger scale spreads more energy;
- rotation reorients an anisotropic Gaussian;
- an opaque front Gaussian occludes the one behind it;
- finite, non-zero gradients reach every parameter group.

An additional test compares the full and chunked compositors.

In [ ]:
# ============================================================
# 5. Renderer correctness tests
# ============================================================

def make_raw_from_valid(
    positions: torch.Tensor,
    scales: torch.Tensor,
    angles: torch.Tensor,
    colors: torch.Tensor,
    opacities: torch.Tensor,
) -> torch.Tensor:
    positions = positions.float()
    scales = scales.float()
    angles = angles.float()
    colors = colors.float()
    opacities = opacities.float()

    return torch.cat(
        [
            inverse_sigmoid(positions),
            inverse_softplus((scales - MIN_SCALE).clamp_min(1e-5)),
            angles.unsqueeze(-1),
            inverse_sigmoid(colors),
            inverse_sigmoid(opacities.unsqueeze(-1)),
        ],
        dim=-1,
    )

def renderer_tests(height: int = 65, width: int = 65) -> pd.DataFrame:
    renderer = GaussianRenderer2D(height, width).to(DEVICE)
    records = []

    # 1. Peak at center
    raw = make_raw_from_valid(
        torch.tensor([[0.5, 0.5]]),
        torch.tensor([[0.08, 0.08]]),
        torch.tensor([0.0]),
        torch.tensor([[1.0, 1.0, 1.0]]),
        torch.tensor([0.8]),
    ).to(DEVICE)
    _, aux = renderer(raw, return_aux=True)
    kernel = aux["kernel"][0]
    max_idx = torch.nonzero(kernel == kernel.max(), as_tuple=False)[0]
    center_idx = torch.tensor([height // 2, width // 2], device=DEVICE)
    passed = torch.max(torch.abs(max_idx - center_idx)).item() <= 1
    records.append(("single Gaussian peaks at center", passed, str(max_idx.tolist())))

    # 2. Isotropic rotation invariance
    raw0 = make_raw_from_valid(
        torch.tensor([[0.5, 0.5]]),
        torch.tensor([[0.10, 0.10]]),
        torch.tensor([0.0]),
        torch.tensor([[1.0, 0.3, 0.2]]),
        torch.tensor([0.8]),
    ).to(DEVICE)
    raw1 = raw0.clone()
    raw1[:, 4] = 1.234
    img0 = renderer(raw0)
    img1 = renderer(raw1)
    diff = (img0 - img1).abs().max().item()
    records.append(("isotropic rotation invariance", diff < 1e-5, f"max diff={diff:.3e}"))

    # 3. Larger scale spreads more energy
    raw_small = make_raw_from_valid(
        torch.tensor([[0.5, 0.5]]),
        torch.tensor([[0.04, 0.04]]),
        torch.tensor([0.0]),
        torch.tensor([[1.0, 1.0, 1.0]]),
        torch.tensor([0.8]),
    ).to(DEVICE)
    raw_large = make_raw_from_valid(
        torch.tensor([[0.5, 0.5]]),
        torch.tensor([[0.15, 0.15]]),
        torch.tensor([0.0]),
        torch.tensor([[1.0, 1.0, 1.0]]),
        torch.tensor([0.8]),
    ).to(DEVICE)
    _, aux_small = renderer(raw_small, return_aux=True)
    _, aux_large = renderer(raw_large, return_aux=True)
    e_small = aux_small["kernel"].sum().item()
    e_large = aux_large["kernel"].sum().item()
    records.append(("larger scale spreads more energy", e_large > e_small, f"{e_small:.2f} -> {e_large:.2f}"))

    # 4. Rotation reorients anisotropic Gaussian
    raw_h = make_raw_from_valid(
        torch.tensor([[0.5, 0.5]]),
        torch.tensor([[0.18, 0.04]]),
        torch.tensor([0.0]),
        torch.tensor([[1.0, 1.0, 1.0]]),
        torch.tensor([0.8]),
    ).to(DEVICE)
    raw_v = raw_h.clone()
    raw_v[:, 4] = math.pi / 2
    _, aux_h = renderer(raw_h, return_aux=True)
    _, aux_v = renderer(raw_v, return_aux=True)
    kh = aux_h["kernel"][0]
    kv = aux_v["kernel"][0]
    horizontal_h = kh[height // 2].sum().item()
    vertical_h = kh[:, width // 2].sum().item()
    horizontal_v = kv[height // 2].sum().item()
    vertical_v = kv[:, width // 2].sum().item()
    passed = horizontal_h > vertical_h and vertical_v > horizontal_v
    records.append(("rotation reorients anisotropic blob", passed, "axis-energy comparison"))

    # 5. Front-to-back occlusion
    raw_two = make_raw_from_valid(
        torch.tensor([[0.5, 0.5], [0.5, 0.5]]),
        torch.tensor([[0.10, 0.10], [0.10, 0.10]]),
        torch.tensor([0.0, 0.0]),
        torch.tensor([[1.0, 0.0, 0.0], [0.0, 0.0, 1.0]]),
        torch.tensor([1.0 - 1e-5, 1.0 - 1e-5]),
    ).to(DEVICE)
    image = renderer(raw_two)
    center_color = image[:, height // 2, width // 2]
    passed = center_color[0].item() > 0.95 and center_color[2].item() < 0.05
    records.append(("opaque front Gaussian occludes rear Gaussian", passed, str(center_color.detach().cpu().tolist())))

    # 6. Differentiability
    raw_grad = initialize_random_gaussians(6, device=DEVICE).detach().clone().requires_grad_(True)
    image = renderer(raw_grad)
    target = torch.rand_like(image)
    loss = F.mse_loss(image, target)
    loss.backward()
    grad = raw_grad.grad
    groups = {
        "position": grad[:, 0:2],
        "scale": grad[:, 2:4],
        "rotation": grad[:, 4:5],
        "color": grad[:, 5:8],
        "opacity": grad[:, 8:9],
    }
    passed = True
    details = []
    for name, g in groups.items():
        ok = g is not None and torch.isfinite(g).all().item() and (g.abs().sum().item() > 0)
        passed = passed and ok
        details.append(f"{name}:{ok}")
    records.append(("finite non-zero gradients reach all groups", passed, ", ".join(details)))

    # 7. Full vs chunked equivalence
    raw_eq = initialize_random_gaussians(17, device=DEVICE, seed=123).detach().clone().requires_grad_(True)
    full = GaussianRenderer2D(height, width, chunk_size=None).to(DEVICE)
    chunked = GaussianRenderer2D(height, width, chunk_size=5).to(DEVICE)

    out_full = full(raw_eq)
    loss_full = out_full.square().mean()
    grad_full = torch.autograd.grad(loss_full, raw_eq, retain_graph=True)[0]

    out_chunk = chunked(raw_eq)
    loss_chunk = out_chunk.square().mean()
    grad_chunk = torch.autograd.grad(loss_chunk, raw_eq)[0]

    image_diff = (out_full - out_chunk).abs().max().item()
    grad_diff = (grad_full - grad_chunk).abs().max().item()
    passed = image_diff < 2e-5 and grad_diff < 2e-5
    records.append(("full and chunked renderer equivalence", passed, f"image={image_diff:.3e}, grad={grad_diff:.3e}"))

    return pd.DataFrame(records, columns=["Test", "Passed", "Details"])

TEST_RESULTS = renderer_tests()
display(TEST_RESULTS)
assert TEST_RESULTS["Passed"].all(), "At least one renderer test failed."

In [ ]:
# ============================================================
# 6. Visualization helpers
# ============================================================

def tensor_to_image(image: torch.Tensor) -> np.ndarray:
    image = image.detach().cpu().clamp(0, 1)
    if image.ndim == 3 and image.shape[0] in (1, 3):
        return image.permute(1, 2, 0).numpy()
    raise ValueError("Expected CHW image.")

def show_image(image: torch.Tensor, title: str = "", ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(4, 4))
    arr = tensor_to_image(image)
    if arr.shape[-1] == 1:
        ax.imshow(arr[..., 0], cmap="gray", vmin=0, vmax=1)
    else:
        ax.imshow(arr)
    ax.set_title(title)
    ax.axis("off")
    return ax

def add_gaussian_ellipses(
    ax,
    raw_gaussians: torch.Tensor,
    width: int,
    height: int,
    alpha: float = 0.8,
    linewidth: float = 1.0,
):
    params = activate_gaussians(raw_gaussians.detach().cpu())
    pos = params["positions"][0].numpy()
    scales = params["scales"][0].numpy()
    angles = params["angles"][0].numpy()
    colors = params["colors"][0].numpy()

    for (x, y), (sx, sy), theta, color in zip(pos, scales, angles, colors):
        ellipse = Ellipse(
            xy=(x * (width - 1), y * (height - 1)),
            width=4.0 * sx * (width - 1),
            height=4.0 * sy * (height - 1),
            angle=np.degrees(theta),
            fill=False,
            edgecolor=color,
            linewidth=linewidth,
            alpha=alpha,
        )
        ax.add_patch(ellipse)

def plot_optimization_curves(result: OptimizationResult, title: str = ""):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(result.loss_history, label="MSE loss")
    ax.set_xlabel("Iteration")
    ax.set_ylabel("MSE")
    ax.set_yscale("log")
    ax.set_title(title + " — loss")
    ax.grid(True, alpha=0.3)
    plt.show()

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(result.psnr_history, label="PSNR")
    ax.set_xlabel("Iteration")
    ax.set_ylabel("PSNR (dB)")
    ax.set_title(title + " — PSNR")
    ax.grid(True, alpha=0.3)
    plt.show()

In [ ]:
# ============================================================
# 7. Part 1 sanity rendering
# ============================================================

H_SANITY, W_SANITY = 160, 160
sanity_renderer = GaussianRenderer2D(H_SANITY, W_SANITY, background_color=(0.02, 0.02, 0.02)).to(DEVICE)

sanity_raw = make_raw_from_valid(
    positions=torch.tensor([
        [0.25, 0.25],
        [0.68, 0.23],
        [0.50, 0.55],
        [0.25, 0.75],
        [0.75, 0.75],
    ]),
    scales=torch.tensor([
        [0.13, 0.05],
        [0.06, 0.13],
        [0.11, 0.11],
        [0.08, 0.03],
        [0.15, 0.06],
    ]),
    angles=torch.tensor([0.3, -0.7, 0.0, 1.1, -0.25]),
    colors=torch.tensor([
        [1.0, 0.2, 0.2],
        [0.2, 1.0, 0.3],
        [0.2, 0.4, 1.0],
        [1.0, 0.8, 0.1],
        [0.9, 0.2, 0.9],
    ]),
    opacities=torch.tensor([0.85, 0.75, 0.65, 0.9, 0.7]),
).to(DEVICE)

sanity_image = sanity_renderer(sanity_raw)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
show_image(sanity_image, "Rendered image", axes[0])
show_image(sanity_image, "Rendered image with 2σ ellipses", axes[1])
add_gaussian_ellipses(axes[1], sanity_raw, W_SANITY, H_SANITY, linewidth=1.5)
plt.tight_layout()
plt.show()

### Part 1 discussion

The tests above check both the mathematical behavior and the differentiability of the renderer.

- The center test confirms that the Gaussian kernel reaches its maximum near \(\mu\).
- The isotropic test confirms that rotation has no effect when \(s_x=s_y\).
- The scale test confirms that increasing the standard deviations increases spatial coverage.
- The anisotropic test confirms that the rotation angle changes the major axis.
- The occlusion test confirms that the implementation uses front-to-back alpha compositing rather than a simple sum.
- The gradient test confirms that positions, scales, angle, color, and opacity all receive usable gradients.
- The equivalence test confirms that chunking preserves both the forward result and backward gradients.

## 5. Dataset loading and preprocessing

The five datasets are:

- MNIST
- FashionMNIST
- CIFAR10
- Olivetti faces
- STL-10

The loading order is:

1. persistent prepared-tensor cache;
2. Hugging Face Datasets for MNIST, FashionMNIST, CIFAR10, and STL-10;
3. torchvision fallback if Hugging Face is unavailable or fails.

Olivetti continues to use `fetch_olivetti_faces`.

Raw downloads are stored on the local Colab filesystem, while the prepared
`DatasetBundle` objects remain in the persistent project cache. All images are
converted to tensors in `[0,1]`. Grayscale datasets are replicated to three
channels so that one RGB renderer can be reused throughout the assignment.

Each run mode selects deterministic subsets. Their indices and prepared image
tensors are cached.


In [ ]:
# ============================================================
# 8. Dataset loading and preprocessing
# ============================================================

_DATASET_MEMORY_CACHE: Dict[str, DatasetBundle] = {}

# Verified Hugging Face schemas.
# Keeping repository IDs and column names together prevents assumptions such as
# using "image" for CIFAR10, whose actual image column is named "img".
HF_DATASET_CONFIG = {
    "MNIST": {
        "repo_id": "ylecun/mnist",
        "image_column": "image",
        "train_split": "train",
        "test_split": "test",
    },
    "FashionMNIST": {
        "repo_id": "zalando-datasets/fashion_mnist",
        "image_column": "image",
        "train_split": "train",
        "test_split": "test",
    },
    "CIFAR10": {
        "repo_id": "uoft-cs/cifar10",
        "image_column": "img",
        "train_split": "train",
        "test_split": "test",
    },
    "STL10": {
        "repo_id": "tanganke/stl10",
        "image_column": "image",
        "train_split": "train",
        "test_split": "test",
    },
}

HF_DATASET_IDS = {
    name: cfg["repo_id"] for name, cfg in HF_DATASET_CONFIG.items()
}

# Raw Hugging Face files are persistent as well. The prepared DatasetBundle
# cache is still checked first, so this directory is touched only on a cache miss.
HF_RAW_CACHE_DIR = CACHE_DIR / "huggingface"
TORCHVISION_RAW_CACHE_DIR = CACHE_DIR / "torchvision"
OLIVETTI_RAW_CACHE_DIR = CACHE_DIR / "olivetti_raw"

for p in [HF_RAW_CACHE_DIR, TORCHVISION_RAW_CACHE_DIR, OLIVETTI_RAW_CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

def _select_indices(length: int, cap: Optional[int], seed: int) -> torch.Tensor:
    gen = torch.Generator().manual_seed(seed)
    perm = torch.randperm(length, generator=gen)
    if cap is None:
        return perm
    return perm[: min(cap, length)]

def _image_to_chw_float(image: Any) -> torch.Tensor:
    """Convert a PIL image or NumPy-compatible image to contiguous float CHW."""
    array = np.asarray(image)
    array = np.ascontiguousarray(array)

    if array.ndim == 2:
        array = array[..., None]
    if array.ndim != 3:
        raise ValueError(f"Expected an HWC or HW image, got shape {array.shape}.")

    if array.shape[-1] not in (1, 3, 4) and array.shape[0] in (1, 3, 4):
        array = np.transpose(array, (1, 2, 0))
        array = np.ascontiguousarray(array)

    if array.shape[-1] == 4:
        array = array[..., :3]

    tensor = torch.from_numpy(array)
    if tensor.dtype == torch.uint8:
        tensor = tensor.float().div_(255.0)
    else:
        tensor = tensor.float()
        if tensor.numel() and tensor.max().item() > 1.0:
            tensor = tensor.div(255.0)

    tensor = tensor.permute(2, 0, 1).contiguous().clamp_(0.0, 1.0)
    if tensor.shape[0] == 1:
        tensor = tensor.repeat(3, 1, 1)
    if tensor.shape[0] != 3:
        raise ValueError(f"Expected 1 or 3 image channels, got {tensor.shape[0]}.")
    return tensor

def _stack_hf_subset(
    split,
    indices: torch.Tensor,
    image_column: str,
) -> torch.Tensor:
    if image_column not in split.column_names:
        raise KeyError(
            f"Expected image column {image_column!r}, but available columns are "
            f"{split.column_names}."
        )

    selected = split.select(indices.tolist())
    images = [
        _image_to_chw_float(example[image_column])
        for example in selected
    ]
    if not images:
        raise ValueError("The selected Hugging Face subset is empty.")
    return torch.stack(images)

def _stack_torchvision_subset(ds, indices: torch.Tensor) -> torch.Tensor:
    images = []
    for idx in indices.tolist():
        image, _ = ds[idx]
        if image.shape[0] == 1:
            image = image.repeat(3, 1, 1)
        images.append(image.float().clamp(0, 1))
    if not images:
        raise ValueError("The selected torchvision subset is empty.")
    return torch.stack(images)

def _build_bundle(
    name: str,
    train_length: int,
    test_length: int,
    train_image_loader,
    test_image_loader,
) -> DatasetBundle:
    train_indices = _select_indices(
        train_length, CFG["train_cap"][name], SEED + 1
    )
    test_indices = _select_indices(
        test_length, CFG["test_cap"][name], SEED + 2
    )
    train_images = train_image_loader(train_indices)
    test_images = test_image_loader(test_indices)

    if tuple(train_images.shape[1:]) != tuple(test_images.shape[1:]):
        raise ValueError(
            f"{name}: train/test image shapes differ: "
            f"{tuple(train_images.shape[1:])} vs {tuple(test_images.shape[1:])}"
        )

    return DatasetBundle(
        name=name,
        train_images=train_images,
        test_images=test_images,
        image_shape=tuple(train_images.shape[1:]),
        train_indices=train_indices,
        test_indices=test_indices,
    )

def _load_hf_splits(name: str):
    if not HF_DATASETS_AVAILABLE or hf_load_dataset is None:
        raise ImportError(
            "The 'datasets' package is unavailable. "
            "Install it with: pip install -q datasets"
        )

    cfg = HF_DATASET_CONFIG[name]
    repo_id = cfg["repo_id"]

    dataset_dict = hf_load_dataset(
        repo_id,
        cache_dir=str(HF_RAW_CACHE_DIR),
    )

    required_splits = {cfg["train_split"], cfg["test_split"]}
    missing = required_splits - set(dataset_dict.keys())
    if missing:
        raise ValueError(
            f"{repo_id} is missing required split(s): {sorted(missing)}"
        )

    train_split = dataset_dict[cfg["train_split"]]
    test_split = dataset_dict[cfg["test_split"]]

    for split_name, split in [
        (cfg["train_split"], train_split),
        (cfg["test_split"], test_split),
    ]:
        if cfg["image_column"] not in split.column_names:
            raise KeyError(
                f"{repo_id}/{split_name}: expected image column "
                f"{cfg['image_column']!r}; available columns are "
                f"{split.column_names}."
            )

    return train_split, test_split

def _load_bundle_from_huggingface(name: str) -> DatasetBundle:
    cfg = HF_DATASET_CONFIG[name]
    train_split, test_split = _load_hf_splits(name)
    image_column = cfg["image_column"]

    return _build_bundle(
        name=name,
        train_length=len(train_split),
        test_length=len(test_split),
        train_image_loader=lambda idx: _stack_hf_subset(
            train_split, idx, image_column
        ),
        test_image_loader=lambda idx: _stack_hf_subset(
            test_split, idx, image_column
        ),
    )

def load_mnist_hf() -> DatasetBundle:
    return _load_bundle_from_huggingface("MNIST")

def load_fashion_mnist_hf() -> DatasetBundle:
    return _load_bundle_from_huggingface("FashionMNIST")

def load_cifar10_hf() -> DatasetBundle:
    return _load_bundle_from_huggingface("CIFAR10")

def load_stl10_hf() -> DatasetBundle:
    return _load_bundle_from_huggingface("STL10")

HF_BUNDLE_LOADERS = {
    "MNIST": load_mnist_hf,
    "FashionMNIST": load_fashion_mnist_hf,
    "CIFAR10": load_cifar10_hf,
    "STL10": load_stl10_hf,
}

def _load_bundle_from_torchvision(name: str) -> DatasetBundle:
    tfm = transforms.ToTensor()

    if name == "MNIST":
        train_ds = torchvision_datasets.MNIST(
            TORCHVISION_RAW_CACHE_DIR,
            train=True,
            download=True,
            transform=tfm,
        )
        test_ds = torchvision_datasets.MNIST(
            TORCHVISION_RAW_CACHE_DIR,
            train=False,
            download=True,
            transform=tfm,
        )
    elif name == "FashionMNIST":
        train_ds = torchvision_datasets.FashionMNIST(
            TORCHVISION_RAW_CACHE_DIR,
            train=True,
            download=True,
            transform=tfm,
        )
        test_ds = torchvision_datasets.FashionMNIST(
            TORCHVISION_RAW_CACHE_DIR,
            train=False,
            download=True,
            transform=tfm,
        )
    elif name == "CIFAR10":
        train_ds = torchvision_datasets.CIFAR10(
            TORCHVISION_RAW_CACHE_DIR,
            train=True,
            download=True,
            transform=tfm,
        )
        test_ds = torchvision_datasets.CIFAR10(
            TORCHVISION_RAW_CACHE_DIR,
            train=False,
            download=True,
            transform=tfm,
        )
    elif name == "STL10":
        train_ds = torchvision_datasets.STL10(
            TORCHVISION_RAW_CACHE_DIR,
            split="train",
            download=True,
            transform=tfm,
        )
        test_ds = torchvision_datasets.STL10(
            TORCHVISION_RAW_CACHE_DIR,
            split="test",
            download=True,
            transform=tfm,
        )
    else:
        raise ValueError(f"torchvision fallback is unsupported for {name}.")

    return _build_bundle(
        name=name,
        train_length=len(train_ds),
        test_length=len(test_ds),
        train_image_loader=lambda idx: _stack_torchvision_subset(train_ds, idx),
        test_image_loader=lambda idx: _stack_torchvision_subset(test_ds, idx),
    )

def _download_olivetti_mat_with_fallback() -> Path:
    """Download the official scikit-learn Olivetti MAT file once.

    scikit-learn's current Figshare endpoint can return HTTP 403 in some Colab
    runtimes. This fallback tries the original NYU mirror first and Figshare
    second, validates the official SHA-256 checksum, and stores the raw MAT file
    in the persistent cache.
    """
    import urllib.request

    destination = OLIVETTI_RAW_CACHE_DIR / "olivettifaces.mat"
    expected_sha256 = (
        "b612fb967f2dc77c9c62d3e1266e0c73"
        "d5fca46a4b8906c18e454d41af987794"
    )

    def _sha256(path: Path) -> str:
        digest = hashlib.sha256()
        with path.open("rb") as f:
            for block in iter(lambda: f.read(1024 * 1024), b""):
                digest.update(block)
        return digest.hexdigest()

    if destination.exists():
        if _sha256(destination) == expected_sha256:
            print(f"[dataset raw cache hit] Olivetti: {destination}")
            return destination
        destination.unlink()

    urls = [
        "https://cs.nyu.edu/~roweis/data/olivettifaces.mat",
        "https://ndownloader.figshare.com/files/5976027",
    ]

    last_error = None
    temporary = destination.with_suffix(".mat.part")

    for url in urls:
        try:
            print(f"[dataset download] Olivetti: {url}")
            request = urllib.request.Request(
                url,
                headers={
                    "User-Agent": (
                        "Mozilla/5.0 CV-Assignment-3 "
                        "Olivetti-dataset-downloader"
                    )
                },
            )
            with urllib.request.urlopen(request, timeout=120) as response:
                with temporary.open("wb") as f:
                    while True:
                        chunk = response.read(1024 * 1024)
                        if not chunk:
                            break
                        f.write(chunk)

            actual_sha256 = _sha256(temporary)
            if actual_sha256 != expected_sha256:
                raise ValueError(
                    "Olivetti checksum mismatch: "
                    f"expected {expected_sha256}, got {actual_sha256}"
                )

            temporary.replace(destination)
            return destination
        except Exception as exc:
            last_error = exc
            if temporary.exists():
                temporary.unlink()
            print(
                f"[dataset download fallback] Olivetti source failed with "
                f"{type(exc).__name__}: {exc}"
            )

    raise RuntimeError(
        "Unable to download the official Olivetti MAT file from either source."
    ) from last_error

def _load_olivetti_images() -> np.ndarray:
    """Load Olivetti through scikit-learn, then use a verified direct fallback."""
    try:
        olivetti = fetch_olivetti_faces(
            data_home=str(OLIVETTI_RAW_CACHE_DIR),
            shuffle=False,
            download_if_missing=True,
        )
        print("[dataset backend] Olivetti: scikit-learn")
        return np.asarray(olivetti.images, dtype=np.float32)
    except Exception as exc:
        print(
            "[dataset fallback] Olivetti: scikit-learn failed with "
            f"{type(exc).__name__}: {exc}"
        )

    from scipy.io import loadmat

    mat_path = _download_olivetti_mat_with_fallback()
    mat = loadmat(mat_path)

    if "faces" not in mat:
        raise KeyError(
            f"Olivetti MAT file does not contain 'faces'; keys are {list(mat)}"
        )

    faces = np.asarray(mat["faces"].T, dtype=np.float32)
    faces -= faces.min()
    max_value = float(faces.max())
    if max_value <= 0:
        raise ValueError("Olivetti images have a non-positive maximum value.")
    faces /= max_value
    faces = faces.reshape((400, 64, 64)).transpose(0, 2, 1)

    print("[dataset backend] Olivetti: verified direct MAT fallback")
    return faces

def _load_olivetti_bundle() -> DatasetBundle:
    all_images_np = _load_olivetti_images()
    all_images = (
        torch.from_numpy(all_images_np)
        .float()
        .unsqueeze(1)
        .repeat(1, 3, 1, 1)
    )

    # Deterministic 80/20 split.
    all_indices = _select_indices(len(all_images), len(all_images), SEED + 99)
    split = int(0.8 * len(all_indices))
    train_pool = all_indices[:split]
    test_pool = all_indices[split:]

    train_indices = train_pool[
        : min(CFG["train_cap"]["Olivetti"], len(train_pool))
    ]
    test_indices = test_pool[
        : min(CFG["test_cap"]["Olivetti"], len(test_pool))
    ]
    train_images = all_images[train_indices]
    test_images = all_images[test_indices]

    return DatasetBundle(
        name="Olivetti",
        train_images=train_images,
        test_images=test_images,
        image_shape=tuple(train_images.shape[1:]),
        train_indices=train_indices,
        test_indices=test_indices,
    )

def load_dataset_bundle(name: str) -> DatasetBundle:
    cache_payload = {
        "name": name,
        "run_mode": RUN_MODE,
        "train_cap": CFG["train_cap"][name],
        "test_cap": CFG["test_cap"][name],
        "seed": SEED,
        # Keep version 2 so valid prepared caches created by the previous
        # notebook remain reusable. Backend choice does not affect this key.
        "version": 2,
    }
    signature = config_hash(cache_payload)
    cache_path = CACHE_DIR / "datasets" / f"{name}_{signature}.pt"

    if name in _DATASET_MEMORY_CACHE:
        return _DATASET_MEMORY_CACHE[name]

    if USE_CACHE and cache_path.exists() and not FORCE_RECOMPUTE:
        print(f"[dataset cache hit] {name}: {cache_path}")
        bundle = load_torch(cache_path)
        _DATASET_MEMORY_CACHE[name] = bundle
        return bundle

    print(f"[dataset cache miss] {name}: preparing and saving persistent cache")

    if name == "Olivetti":
        bundle = _load_olivetti_bundle()
    elif name in HF_BUNDLE_LOADERS:
        try:
            cfg = HF_DATASET_CONFIG[name]
            print(
                f"[dataset backend] {name}: Hugging Face "
                f"({cfg['repo_id']}, image_column={cfg['image_column']!r})"
            )
            bundle = HF_BUNDLE_LOADERS[name]()
        except Exception as exc:
            print(
                f"[dataset fallback] {name}: Hugging Face failed with "
                f"{type(exc).__name__}: {exc}"
            )
            print(f"[dataset backend] {name}: torchvision fallback")
            bundle = _load_bundle_from_torchvision(name)
    else:
        raise ValueError(f"Unknown dataset: {name}")

    save_torch(bundle, cache_path)
    _DATASET_MEMORY_CACHE[name] = bundle
    print(f"[dataset cache saved] {name}: {cache_path}")
    return bundle

DATASET_NAMES = ["MNIST", "FashionMNIST", "CIFAR10", "Olivetti", "STL10"]

# Load lazily in later sections. This summary loads each dataset once.
dataset_summary = []
for dataset_name in DATASET_NAMES:
    bundle = load_dataset_bundle(dataset_name)
    dataset_summary.append({
        "Dataset": dataset_name,
        "Train images": len(bundle.train_images),
        "Test images": len(bundle.test_images),
        "Shape": bundle.image_shape,
    })
display(pd.DataFrame(dataset_summary))

In [ ]:
# Persistent-cache inventory
def directory_size_bytes(path: Path) -> int:
    if not path.exists():
        return 0
    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file())

cache_inventory_rows = []
for label, path in [
    ("Raw dataset downloads (local in Colab)", DATA_DIR),
    ("Prepared dataset cache", CACHE_DIR / "datasets"),
    ("PCA cache", CACHE_DIR / "pca"),
    ("Predictor checkpoints", CHECKPOINT_DIR / "predictors"),
    ("Experiment results", RESULTS_DIR),
]:
    cache_inventory_rows.append({
        "Artifact group": label,
        "Path": str(path),
        "Files": sum(1 for p in path.rglob("*") if p.is_file()) if path.exists() else 0,
        "Size (MB)": directory_size_bytes(path) / (1024 ** 2),
    })

display(pd.DataFrame(cache_inventory_rows))

In [ ]:
# Display one image from each dataset
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, dataset_name in zip(axes, DATASET_NAMES):
    bundle = load_dataset_bundle(dataset_name)
    show_image(bundle.train_images[0], dataset_name, ax)
plt.tight_layout()
plt.show()

## 6. Shared Gaussian optimization engine

The same optimization function is reused for:

- fixed-\(N\) fitting in Part 2;
- adaptive fitting in Part 3;
- random initialization in Part 5;
- PCA-predicted initialization in Part 5.

The function records MSE, PSNR, Gaussian count, selected snapshots, and runtime in one pass.

In [ ]:
# ============================================================
# 9. Shared optimization engine
# ============================================================

def optimize_gaussians(
    target: torch.Tensor,
    initial_raw_gaussians: torch.Tensor,
    renderer: GaussianRenderer2D,
    num_iterations: int,
    learning_rate: float = 0.03,
    snapshot_iterations: Optional[List[int]] = None,
    density_controller: Optional[Any] = None,
    verbose: bool = False,
) -> OptimizationResult:
    target = target.to(DEVICE, DTYPE)
    raw = nn.Parameter(initial_raw_gaussians.detach().clone().to(DEVICE, DTYPE))
    optimizer = torch.optim.Adam([raw], lr=learning_rate)

    if snapshot_iterations is None:
        snapshot_iterations = make_snapshot_iterations(num_iterations, CFG["snapshot_count"])
    snapshot_set = set(snapshot_iterations)

    loss_history = []
    psnr_history = []
    count_history = []
    snapshots = {}

    start_time = time.perf_counter()

    for iteration in range(num_iterations):
        optimizer.zero_grad(set_to_none=True)
        reconstruction = renderer(raw)
        loss = F.mse_loss(reconstruction, target)
        loss.backward()

        if density_controller is not None:
            density_controller.accumulate(raw)

        optimizer.step()

        with torch.no_grad():
            mse_value = float(loss.detach().cpu())
            psnr_value = float(mse_to_psnr(loss.detach()).cpu())
            loss_history.append(mse_value)
            psnr_history.append(psnr_value)
            count_history.append(int(raw.shape[-2]))

            if iteration in snapshot_set:
                snapshots[iteration] = {
                    "image": reconstruction.detach().cpu(),
                    "raw_gaussians": raw.detach().cpu(),
                }

        if density_controller is not None and density_controller.should_update(iteration):
            raw, optimizer = density_controller.apply(
                raw=raw,
                optimizer=optimizer,
                iteration=iteration,
            )

        if verbose and (iteration % max(1, num_iterations // 10) == 0):
            print(
                f"iter={iteration:4d} "
                f"N={raw.shape[-2]:4d} "
                f"loss={loss_history[-1]:.6f} "
                f"PSNR={psnr_history[-1]:.2f}"
            )

    with torch.no_grad():
        final_image = renderer(raw).detach().cpu()

    runtime = time.perf_counter() - start_time

    return OptimizationResult(
        final_raw_gaussians=raw.detach().cpu(),
        final_image=final_image,
        loss_history=loss_history,
        psnr_history=psnr_history,
        gaussian_count_history=count_history,
        snapshots=snapshots,
        runtime_seconds=runtime,
    )

# Part 2 — Single-image optimization

For one representative image from each dataset, the Gaussian parameters are initialized randomly and optimized with Adam.

The required capacity ablation evaluates \(N\in\{50,100,200,400\}\).  
In debug mode only a smaller subset is used so that the complete notebook can be tested quickly.

In [ ]:
# ============================================================
# 10. Part 2 experiments
# ============================================================

PART2_RESULTS = {}

def run_part2_experiment(
    dataset_name: str,
    num_gaussians: int,
    image_index: int = 0,
) -> OptimizationResult:
    bundle = load_dataset_bundle(dataset_name)
    target = bundle.test_images[image_index]
    _, height, width = target.shape

    payload = {
        "part": 2,
        "dataset": dataset_name,
        "image_index": image_index,
        "num_gaussians": num_gaussians,
        "iterations": CFG["part2_iterations"],
        "lr": 0.03,
        "seed": SEED,
        "renderer_version": 2,
        "run_mode": RUN_MODE,
    }
    path = RESULTS_DIR / "part2" / f"{dataset_name}_{config_hash(payload)}.pt"

    if USE_CACHE and path.exists() and not FORCE_RECOMPUTE:
        return load_torch(path)

    renderer = GaussianRenderer2D(height, width).to(DEVICE)
    initial = initialize_random_gaussians(
        num_gaussians,
        seed=SEED + image_index + num_gaussians,
        device=DEVICE,
    )
    result = optimize_gaussians(
        target=target,
        initial_raw_gaussians=initial,
        renderer=renderer,
        num_iterations=CFG["part2_iterations"],
        learning_rate=0.03,
        verbose=False,
    )
    save_torch(result, path)
    return result

if ENABLE_PART_2:
    for dataset_name in DATASET_NAMES:
        PART2_RESULTS[dataset_name] = {}
        for n in CFG["part2_capacity_values"]:
            print(f"Part 2: {dataset_name}, N={n}")
            PART2_RESULTS[dataset_name][n] = run_part2_experiment(dataset_name, n)

In [ ]:
# Part 2 visual summary
if ENABLE_PART_2:
    chosen_n = max(CFG["part2_capacity_values"])
    for dataset_name in DATASET_NAMES:
        bundle = load_dataset_bundle(dataset_name)
        target = bundle.test_images[0]
        result = PART2_RESULTS[dataset_name][chosen_n]

        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        show_image(target, f"{dataset_name}: target", axes[0])
        show_image(result.final_image, f"Reconstruction\nPSNR={result.psnr_history[-1]:.2f} dB", axes[1])
        show_image(result.final_image, "Learned 2σ ellipses", axes[2])
        add_gaussian_ellipses(
            axes[2],
            result.final_raw_gaussians,
            target.shape[-1],
            target.shape[-2],
            alpha=0.45,
            linewidth=0.8,
        )
        plt.tight_layout()
        plt.show()

        plot_optimization_curves(result, f"{dataset_name}, N={chosen_n}")

In [ ]:
# Capacity ablation: final PSNR as a function of N
if ENABLE_PART_2:
    rows = []
    for dataset_name, by_n in PART2_RESULTS.items():
        for n, result in by_n.items():
            rows.append({
                "Dataset": dataset_name,
                "N": n,
                "Final PSNR": result.psnr_history[-1],
                "Runtime (s)": result.runtime_seconds,
            })

    part2_table = pd.DataFrame(rows)
    display(part2_table)

    if len(CFG["part2_capacity_values"]) > 1:
        fig, ax = plt.subplots(figsize=(8, 5))
        for dataset_name in DATASET_NAMES:
            subset = part2_table[part2_table["Dataset"] == dataset_name].sort_values("N")
            ax.plot(subset["N"], subset["Final PSNR"], marker="o", label=dataset_name)
        ax.set_xlabel("Number of Gaussians, N")
        ax.set_ylabel("Final PSNR (dB)")
        ax.set_title("Part 2 capacity ablation")
        ax.grid(True, alpha=0.3)
        ax.legend()
        plt.show()

### Part 2 discussion

The capacity ablation shows a consistent improvement in reconstruction quality as the number of Gaussians increases from 50 to 400 on all five datasets.

For MNIST, the final PSNR increases from 27.59 dB with 50 Gaussians to 41.94 dB with 400. FashionMNIST improves from 25.19 dB to 45.99 dB, CIFAR10 from 29.40 dB to 52.16 dB, Olivetti from 25.63 dB to 39.22 dB, and STL10 from 20.75 dB to 29.13 dB.

The experiment therefore confirms that additional Gaussians provide useful representational capacity. The improvement is especially large for CIFAR10 and FashionMNIST. STL10 remains the most difficult dataset because its native 96×96 resolution and complex natural-image content require more primitives to represent fine details.

Within the tested range, the results do not yet show clear saturation: PSNR continues to improve at N=400. Saturation is nevertheless expected at a larger capacity because the image resolution is finite, additional Gaussians eventually become redundant, and a fixed optimization budget may not be sufficient to use every added primitive effectively.

The ellipse visualizations show that many Gaussians become small and concentrate around object boundaries, strokes, facial structures, and textured regions. These locations produce larger reconstruction errors and therefore stronger gradients. Flat regions can be represented by fewer, broader Gaussians.

Although the tested range does not yet exhibit clear saturation, the marginal improvement between successive capacities gradually decreases, suggesting diminishing returns as model capacity grows.

# Part 3 — Adaptive density control

The adaptive method starts from a small set of Gaussians and changes the topology every \(M\) iterations.

For each Gaussian, the average positional gradient is accumulated:

\[
\bar g_i =
\left\|
\frac{1}{M}
\sum_t
\frac{\partial L}{\partial \mu_i}
\right\|.
\]

Rules:

- prune if opacity is below a threshold;
- clone if gradient is high and scale is small;
- split if gradient is high and scale is large;
- cap the total number of Gaussians.

The implementation migrates Adam state from the old parameter tensor to the new tensor. Surviving Gaussians retain their moments. Newly created children start with fresh moments.

In [ ]:
# ============================================================
# 11. Adaptive density controller
# ============================================================

class DensityController:
    def __init__(
        self,
        interval: int = 50,
        opacity_threshold: float = 0.03,
        gradient_threshold: float = 5e-4,
        scale_threshold: float = 0.08,
        split_factor: float = 1.6,
        max_gaussians: int = 400,
        start_iteration: int = 20,
    ):
        self.interval = interval
        self.opacity_threshold = opacity_threshold
        self.gradient_threshold = gradient_threshold
        self.scale_threshold = scale_threshold
        self.split_factor = split_factor
        self.max_gaussians = max_gaussians
        self.start_iteration = start_iteration

        self.grad_sum = None
        self.grad_steps = 0

    def accumulate(self, raw: nn.Parameter):
        if raw.grad is None:
            return
        # Convert raw-position gradients to a stable per-Gaussian score.
        g = raw.grad.detach()[..., 0:2]
        if raw.ndim == 3:
            g = g.mean(dim=0)
        g = torch.linalg.vector_norm(g, dim=-1)

        if self.grad_sum is None or self.grad_sum.shape != g.shape:
            self.grad_sum = torch.zeros_like(g)
            self.grad_steps = 0

        self.grad_sum = self.grad_sum + g
        self.grad_steps += 1

    def should_update(self, iteration: int) -> bool:
        return (
            iteration >= self.start_iteration
            and (iteration + 1) % self.interval == 0
            and self.grad_steps > 0
        )

    @staticmethod
    def _migrate_adam_state(
        old_optimizer: torch.optim.Optimizer,
        old_param: nn.Parameter,
        new_param: nn.Parameter,
        source_indices: torch.Tensor,
        learning_rate: float,
    ) -> torch.optim.Optimizer:
        new_optimizer = torch.optim.Adam([new_param], lr=learning_rate)
        old_state = old_optimizer.state.get(old_param, None)

        if not old_state:
            return new_optimizer

        new_state = new_optimizer.state[new_param]
        for key, value in old_state.items():
            if torch.is_tensor(value) and value.shape == old_param.shape:
                migrated = torch.zeros_like(new_param.data)
                valid = source_indices >= 0
                migrated[valid] = value[source_indices[valid]]
                new_state[key] = migrated
            elif torch.is_tensor(value):
                new_state[key] = value.detach().clone()
            else:
                new_state[key] = value
        return new_optimizer

    def apply(
        self,
        raw: nn.Parameter,
        optimizer: torch.optim.Optimizer,
        iteration: int,
    ) -> Tuple[nn.Parameter, torch.optim.Optimizer]:
        with torch.no_grad():
            raw_data = raw.detach()
            params = activate_gaussians(raw_data)
            opacities = params["opacities"][0]
            scales = params["scales"][0]
            mean_scale = scales.mean(dim=-1)

            grad_score = self.grad_sum / max(self.grad_steps, 1)

            keep_mask = opacities >= self.opacity_threshold
            if keep_mask.sum() == 0:
                keep_mask[torch.argmax(opacities)] = True

            kept_indices = torch.nonzero(keep_mask, as_tuple=False).flatten()
            kept_raw = raw_data[kept_indices]
            kept_grad = grad_score[kept_indices]
            kept_scale = mean_scale[kept_indices]

            high_grad = kept_grad >= self.gradient_threshold
            clone_mask = high_grad & (kept_scale < self.scale_threshold)
            split_mask = high_grad & (kept_scale >= self.scale_threshold)

            rows = []
            sources = []

            for local_idx, old_idx in enumerate(kept_indices.tolist()):
                parent = kept_raw[local_idx]
                if split_mask[local_idx]:
                    valid = activate_gaussians(parent.unsqueeze(0))
                    pos = valid["positions"][0, 0]
                    scale = valid["scales"][0, 0]
                    angle = valid["angles"][0, 0]
                    color = valid["colors"][0, 0]
                    opacity = valid["opacities"][0, 0]

                    direction = torch.tensor(
                        [torch.cos(angle), torch.sin(angle)],
                        device=raw.device,
                        dtype=raw.dtype,
                    )
                    offset = direction * (0.35 * scale.max())
                    child_scale = (scale / self.split_factor).clamp_min(MIN_SCALE * 1.1)

                    for sign in (-1.0, 1.0):
                        child_pos = (pos + sign * offset).clamp(1e-4, 1 - 1e-4)
                        child = make_raw_from_valid(
                            child_pos.unsqueeze(0).cpu(),
                            child_scale.unsqueeze(0).cpu(),
                            angle.unsqueeze(0).cpu(),
                            color.unsqueeze(0).cpu(),
                            opacity.unsqueeze(0).cpu(),
                        ).to(raw.device)
                        rows.append(child[0])
                        sources.append(-1)

                else:
                    rows.append(parent)
                    sources.append(old_idx)

                    if clone_mask[local_idx]:
                        valid = activate_gaussians(parent.unsqueeze(0))
                        pos = valid["positions"][0, 0]
                        scale = valid["scales"][0, 0]
                        angle = valid["angles"][0, 0]
                        color = valid["colors"][0, 0]
                        opacity = valid["opacities"][0, 0]

                        noise = torch.randn_like(pos) * (0.15 * scale.mean())
                        clone_pos = (pos + noise).clamp(1e-4, 1 - 1e-4)
                        clone = make_raw_from_valid(
                            clone_pos.unsqueeze(0).cpu(),
                            scale.unsqueeze(0).cpu(),
                            angle.unsqueeze(0).cpu(),
                            color.unsqueeze(0).cpu(),
                            opacity.unsqueeze(0).cpu(),
                        ).to(raw.device)
                        rows.append(clone[0])
                        sources.append(-1)

            if len(rows) > self.max_gaussians:
                rows = rows[: self.max_gaussians]
                sources = sources[: self.max_gaussians]

            new_data = torch.stack(rows, dim=0)
            new_param = nn.Parameter(new_data)
            source_indices = torch.tensor(sources, device=raw.device, dtype=torch.long)
            lr = optimizer.param_groups[0]["lr"]
            new_optimizer = self._migrate_adam_state(
                optimizer, raw, new_param, source_indices, lr
            )

            self.grad_sum = torch.zeros(new_param.shape[0], device=raw.device)
            self.grad_steps = 0

            return new_param, new_optimizer

In [ ]:
# ============================================================
# 12. Part 3 experiments
# ============================================================

PART3_RESULTS = {}

def run_part3_experiment(dataset_name: str, image_index: int = 0) -> OptimizationResult:
    bundle = load_dataset_bundle(dataset_name)
    target = bundle.test_images[image_index]
    _, height, width = target.shape

    payload = {
        "part": 3,
        "dataset": dataset_name,
        "image_index": image_index,
        "iterations": CFG["part3_iterations"],
        "interval": CFG["part3_interval"],
        "max_gaussians": CFG["part3_max_gaussians"],
        "seed": SEED,
        "version": 2,
        "run_mode": RUN_MODE,
    }
    path = RESULTS_DIR / "part3" / f"{dataset_name}_{config_hash(payload)}.pt"

    if USE_CACHE and path.exists() and not FORCE_RECOMPUTE:
        return load_torch(path)

    renderer = GaussianRenderer2D(height, width).to(DEVICE)
    initial = initialize_random_gaussians(
        10,
        seed=SEED + image_index + 300,
        device=DEVICE,
        initial_scale=0.15,
        initial_opacity=0.35,
    )
    controller = DensityController(
        interval=CFG["part3_interval"],
        opacity_threshold=0.02,
        gradient_threshold=3e-4,
        scale_threshold=0.08,
        split_factor=1.6,
        max_gaussians=CFG["part3_max_gaussians"],
        start_iteration=max(10, CFG["part3_interval"] // 2),
    )
    result = optimize_gaussians(
        target=target,
        initial_raw_gaussians=initial,
        renderer=renderer,
        num_iterations=CFG["part3_iterations"],
        learning_rate=0.025,
        density_controller=controller,
        verbose=False,
    )
    save_torch(result, path)
    return result

if ENABLE_PART_3:
    for dataset_name in DATASET_NAMES:
        print("Part 3:", dataset_name)
        PART3_RESULTS[dataset_name] = run_part3_experiment(dataset_name)

In [ ]:
# Part 3 refinement progress and Gaussian visualizations
if ENABLE_PART_3:
    for dataset_name, result in PART3_RESULTS.items():
        bundle = load_dataset_bundle(dataset_name)
        target = bundle.test_images[0]

        snap_items = sorted(result.snapshots.items())
        cols = len(snap_items) + 1

        fig, axes = plt.subplots(1, cols, figsize=(3 * cols, 3))
        show_image(target, "Target", axes[0])
        for ax, (iteration, snap) in zip(axes[1:], snap_items):
            show_image(
                snap["image"],
                f"it {iteration}\nN={snap['raw_gaussians'].shape[-2]}",
                ax,
            )
        plt.suptitle(f"{dataset_name}: adaptive refinement progress")
        plt.tight_layout()
        plt.show()

        fig, axes = plt.subplots(1, len(snap_items), figsize=(3 * len(snap_items), 3))
        if len(snap_items) == 1:
            axes = [axes]
        for ax, (iteration, snap) in zip(axes, snap_items):
            show_image(snap["image"], f"it {iteration}", ax)
            add_gaussian_ellipses(
                ax,
                snap["raw_gaussians"],
                target.shape[-1],
                target.shape[-2],
                alpha=0.35,
                linewidth=0.6,
            )
        plt.suptitle(f"{dataset_name}: 2σ Gaussian evolution")
        plt.tight_layout()
        plt.show()

        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(result.gaussian_count_history)
        ax.set_xlabel("Iteration")
        ax.set_ylabel("Number of Gaussians")
        ax.set_title(f"{dataset_name}: adaptive Gaussian count")
        ax.grid(True, alpha=0.3)
        plt.show()

        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(result.psnr_history)
        ax.set_xlabel("Iteration")
        ax.set_ylabel("PSNR (dB)")
        ax.set_title(f"{dataset_name}: adaptive PSNR")
        ax.grid(True, alpha=0.3)
        plt.show()

In [ ]:
# Compare adaptive density control with a fixed-N baseline of comparable budget.
if ENABLE_PART_3:
    comparison_rows = []

    for dataset_name, adaptive in PART3_RESULTS.items():
        final_n = adaptive.final_raw_gaussians.shape[-2]
        fixed = run_part2_experiment(dataset_name, int(final_n))

        comparison_rows.append({
            "Dataset": dataset_name,
            "Adaptive N": final_n,
            "Adaptive PSNR": adaptive.psnr_history[-1],
            "Fixed N": fixed.final_raw_gaussians.shape[-2],
            "Fixed PSNR": fixed.psnr_history[-1],
        })

    part3_comparison = pd.DataFrame(comparison_rows)
    display(part3_comparison)

### Part 3 discussion

Adaptive density control produced mixed, dataset-dependent results.

It substantially outperformed the comparable fixed-N baseline on MNIST: 47.43 dB with 106 adaptive Gaussians compared with 34.42 dB for 106 fixed Gaussians. It also improved FashionMNIST from 42.66 dB to 45.03 dB and Olivetti from 23.89 dB to 24.96 dB.

However, adaptive control was slightly worse on CIFAR10, reaching 24.13 dB compared with 24.70 dB for the fixed baseline, and clearly worse on STL10, reaching 20.57 dB compared with 23.18 dB.

These results show that adaptive placement can be highly effective when the image structure is sparse and localized, as in digits and clothing silhouettes. On complex natural images, the gradient and scale thresholds may create too few primitives or may split and clone them at suboptimal times. CIFAR10 finished with only 22 Gaussians, while STL10 finished with 97, which may be insufficient for their texture and spatial complexity.

The visual progression shows how the method starts from 10 primitives and gradually allocates additional Gaussians to high-error regions. Small high-gradient Gaussians are cloned, large high-gradient Gaussians are split, and nearly transparent Gaussians are pruned. Flat regions remain represented by fewer and broader primitives.

The main hyperparameters control a trade-off between stability and responsiveness. A smaller adaptation interval reacts faster but may change the topology before the optimizer stabilizes. A lower opacity threshold preserves more weak Gaussians, while a lower gradient threshold causes more aggressive growth. The scale threshold determines whether a high-gradient primitive is cloned or split, and the cap on N prevents uncontrolled growth.

During topology changes, Adam state is migrated for surviving Gaussians. Their `exp_avg` and `exp_avg_sq` values are retained, while newly created Gaussians start with zero moments.

# Part 4 — PCA/eigenspace initialization

For each dataset:

1. Flatten the training images.
2. Fit PCA.
3. Store the mean, principal components, explained variance, and coefficients.
4. Visualize cumulative explained variance.
5. Visualize leading eigenimages.
6. Reconstruct examples from the retained coefficients.
7. Train an MLP that maps PCA coefficients to \(N\times9\) Gaussian parameters.
8. Evaluate one-shot reconstruction quality on held-out images.

The MLP receives no ground-truth Gaussian labels. It is trained only through the differentiable renderer using pixel reconstruction loss.

In [ ]:
# ============================================================
# 13. PCA pipeline
# ============================================================

PCA_ARTIFACTS = {}

def fit_or_load_pca(dataset_name: str) -> Dict[str, Any]:
    bundle = load_dataset_bundle(dataset_name)
    images = bundle.train_images[: CFG["pca_cap"][dataset_name]]
    flat = images.flatten(1).numpy()

    max_components = min(
        CFG["pca_k_cap"],
        flat.shape[0] - 1,
        flat.shape[1],
    )
    payload = {
        "dataset": dataset_name,
        "n_samples": len(flat),
        "max_components": max_components,
        "shape": bundle.image_shape,
        "seed": SEED,
        "run_mode": RUN_MODE,
        "version": 2,
    }
    path = CACHE_DIR / "pca" / f"{dataset_name}_{config_hash(payload)}.pt"

    if USE_CACHE and path.exists() and not FORCE_RECOMPUTE:
        return load_torch(path)

    solver = "randomized" if max_components < min(flat.shape) else "full"
    pca = PCA(
        n_components=max_components,
        svd_solver=solver,
        random_state=SEED,
    )
    train_coeff = pca.fit_transform(flat)
    test_coeff = pca.transform(bundle.test_images.flatten(1).numpy())

    cumulative = np.cumsum(pca.explained_variance_ratio_)
    k90 = int(np.searchsorted(cumulative, 0.90) + 1)
    k90 = min(k90, max_components)

    artifact = {
        "dataset": dataset_name,
        "mean": torch.from_numpy(pca.mean_).float(),
        "components": torch.from_numpy(pca.components_).float(),
        "explained_variance_ratio": torch.from_numpy(pca.explained_variance_ratio_).float(),
        "cumulative_explained_variance": torch.from_numpy(cumulative).float(),
        "train_coefficients": torch.from_numpy(train_coeff).float(),
        "test_coefficients": torch.from_numpy(test_coeff).float(),
        "image_shape": bundle.image_shape,
        "k90": k90,
        "max_components": max_components,
    }
    save_torch(artifact, path)
    return artifact

if ENABLE_PCA:
    for dataset_name in DATASET_NAMES:
        print("PCA:", dataset_name)
        PCA_ARTIFACTS[dataset_name] = fit_or_load_pca(dataset_name)

In [ ]:
if EXECUTION_STAGE == "prepare":
    print("=" * 72)
    print("CPU preparation stage completed.")
    print("Datasets and PCA artifacts are stored in the persistent workspace.")
    print("Next: switch Colab to GPU, set EXECUTION_STAGE='pipeline',")
    print("keep the same RUN_MODE, and run the notebook again.")
    print("=" * 72)

### PCA analysis

**Important note:** STL10 did not reach 90% explained variance within the configured maximum of 256 components. Its cumulative explained variance at K=256 is only 87.76%. Therefore, the true number of components required for 90% is greater than 256.


In [ ]:
# Cumulative explained variance curves
if ENABLE_PCA:
    fig, ax = plt.subplots(figsize=(9, 6))
    pca_rows = []

    for dataset_name in DATASET_NAMES:
        artifact = PCA_ARTIFACTS[dataset_name]
        cumulative = artifact["cumulative_explained_variance"].numpy()
        ax.plot(np.arange(1, len(cumulative) + 1), cumulative, label=dataset_name)
        pca_rows.append({
            "Dataset": dataset_name,
            "Available components": artifact["max_components"],
            "Components for ~90% variance": artifact["k90"],
            "Variance reached": float(cumulative[-1]),
        })

    ax.axhline(0.90, linestyle="--", linewidth=1)
    ax.set_xlabel("Number of PCA components")
    ax.set_ylabel("Cumulative explained variance")
    ax.set_title("PCA cumulative explained variance")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()

    pca_summary = pd.DataFrame(pca_rows)
    display(pca_summary)

PCA analysis
The PCA results reveal clear differences in dataset complexity.

Olivetti requires the fewest components to reach approximately 90% explained variance: 62 components. This is consistent with the aligned structure of the face dataset, where eyes, noses, mouths, and facial boundaries appear in similar spatial locations.

FashionMNIST requires 83 components and MNIST requires 86. Their images contain relatively simple foreground objects on uniform backgrounds, although variations in shape, pose, and stroke structure still require multiple components.

CIFAR10 requires 97 components. Its natural images contain more variation in color, object position, background, and texture than the digit and clothing datasets.

STL10 is the most difficult dataset. Even 256 components explain only 87.76% of its variance, so the true K required for 90% is greater than the configured PCA limit. This is consistent with its higher 96×96 resolution and greater natural-image diversity.

The eigenimages reflect these differences. The leading components of MNIST, FashionMNIST, and Olivetti contain recognizable global structures, while the natural-image eigenimages are more diffuse and encode broad color and spatial variation rather than a single clear object pattern.

### PCA analysis

The PCA results reveal clear differences in dataset complexity.

Olivetti requires the fewest components to reach approximately 90% explained variance: 62 components. This is consistent with the aligned structure of the face dataset, where eyes, noses, mouths, and facial boundaries appear in similar spatial locations.

FashionMNIST requires 83 components and MNIST requires 86. Their images contain relatively simple foreground objects on uniform backgrounds, although variations in shape, pose, and stroke structure still require multiple components.

CIFAR10 requires 97 components. Its natural images contain more variation in color, object position, background, and texture than the digit and clothing datasets.

STL10 is the most difficult dataset. Even 256 components explain only 87.76% of its variance, so the true K required for 90% is greater than the configured PCA limit. This is consistent with its higher 96×96 resolution and greater natural-image diversity.

The eigenimages reflect these differences. The leading components of MNIST, FashionMNIST, and Olivetti contain recognizable global structures, while the natural-image eigenimages are more diffuse and encode broad color and spatial variation rather than a single clear object pattern.

In [ ]:
# Leading eigenimages and PCA reconstructions
if ENABLE_PCA:
    for dataset_name in DATASET_NAMES:
        artifact = PCA_ARTIFACTS[dataset_name]
        bundle = load_dataset_bundle(dataset_name)
        c, h, w = artifact["image_shape"]

        n_eigen = min(8, artifact["components"].shape[0])
        fig, axes = plt.subplots(2, 4, figsize=(12, 6))
        for idx, ax in enumerate(axes.flat):
            if idx < n_eigen:
                eig = artifact["components"][idx].reshape(c, h, w)
                # Per-image normalization is used only for visualization.
                eig_vis = (eig - eig.min()) / (eig.max() - eig.min() + 1e-8)
                show_image(eig_vis, f"PC {idx + 1}", ax)
            else:
                ax.axis("off")
        plt.suptitle(f"{dataset_name}: leading eigenimages")
        plt.tight_layout()
        plt.show()

        k = min(artifact["k90"], artifact["max_components"])
        coeff = artifact["test_coefficients"][:4, :k]
        components = artifact["components"][:k]
        recon_flat = artifact["mean"] + coeff @ components
        recon = recon_flat.reshape(-1, c, h, w).clamp(0, 1)

        fig, axes = plt.subplots(4, 2, figsize=(6, 12))
        for row in range(4):
            show_image(bundle.test_images[row], "Original", axes[row, 0])
            show_image(recon[row], f"PCA reconstruction, K={k}", axes[row, 1])
        plt.suptitle(dataset_name)
        plt.tight_layout()
        plt.show()

### PCA interpretation

Digit and clothing datasets are expected to need fewer components than natural-image datasets because their images have simpler structure and lower visual diversity. CIFAR10 and STL-10 contain varied objects, backgrounds, textures, and colors, so their variance is spread over more directions. Aligned face images are especially favorable for PCA because corresponding facial structures appear at similar image locations.

In [ ]:
# ============================================================
# 14. PCA-to-Gaussian predictor
# ============================================================

class PCAToGaussianMLP(nn.Module):
    def __init__(self, input_dim: int, num_gaussians: int):
        super().__init__()
        self.input_dim = input_dim
        self.num_gaussians = num_gaussians
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 9 * num_gaussians),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.network(z).view(z.shape[0], self.num_gaussians, 9)

def predictor_num_gaussians(dataset_name: str) -> int:
    if RUN_MODE == "debug":
        return 50
    if dataset_name in ("MNIST", "FashionMNIST"):
        return 100
    if dataset_name == "CIFAR10":
        return 200
    if dataset_name == "Olivetti":
        return 200
    return 300

In [ ]:
# ============================================================
# 15. Predictor training and checkpointing
# ============================================================

PREDICTORS = {}
PREDICTOR_HISTORIES = {}

def train_or_load_predictor(dataset_name: str):
    artifact = PCA_ARTIFACTS[dataset_name]
    bundle = load_dataset_bundle(dataset_name)

    k = min(artifact["k90"], artifact["max_components"])
    n = predictor_num_gaussians(dataset_name)
    c, h, w = bundle.image_shape

    payload = {
        "dataset": dataset_name,
        "k": k,
        "n": n,
        "epochs": CFG["predictor_epochs"],
        "batch_size": CFG["predictor_batch_size"],
        "run_mode": RUN_MODE,
        "seed": SEED,
        "version": 2,
    }
    path = CHECKPOINT_DIR / "predictors" / f"{dataset_name}_{config_hash(payload)}.pt"

    model = PCAToGaussianMLP(k, n).to(DEVICE)
    if USE_CACHE and path.exists() and not FORCE_RECOMPUTE:
        checkpoint = load_torch(path)
        model.load_state_dict(checkpoint["model_state_dict"])
        return model, checkpoint["history"], k, n

    z_train = artifact["train_coefficients"][:, :k]
    images = bundle.train_images[: len(z_train)]
    train_ds = TensorDataset(z_train, images)

    batch_size = min(CFG["predictor_batch_size"], len(train_ds))
    loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )

    # STL-10 can require chunking to reduce memory.
    chunk_size = 64 if dataset_name == "STL10" else None
    renderer = GaussianRenderer2D(h, w, chunk_size=chunk_size).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    history = {"train_loss": [], "train_psnr": []}
    best_loss = float("inf")
    best_state = None

    for epoch in range(CFG["predictor_epochs"]):
        model.train()
        epoch_losses = []

        for z_batch, image_batch in loader:
            z_batch = z_batch.to(DEVICE, DTYPE)
            image_batch = image_batch.to(DEVICE, DTYPE)

            optimizer.zero_grad(set_to_none=True)
            predicted_raw = model(z_batch)
            reconstruction = renderer(predicted_raw)
            loss = F.mse_loss(reconstruction, image_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            epoch_losses.append(float(loss.detach().cpu()))

        mean_loss = float(np.mean(epoch_losses))
        mean_psnr = float(-10.0 * np.log10(max(mean_loss, 1e-12)))
        history["train_loss"].append(mean_loss)
        history["train_psnr"].append(mean_psnr)

        if mean_loss < best_loss:
            best_loss = mean_loss
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

        print(
            f"{dataset_name}: epoch {epoch + 1}/{CFG['predictor_epochs']} "
            f"loss={mean_loss:.6f}, PSNR={mean_psnr:.2f}"
        )

    model.load_state_dict(best_state)
    save_torch(
        {
            "model_state_dict": best_state,
            "history": history,
            "k": k,
            "n": n,
            "payload": payload,
        },
        path,
    )
    return model, history, k, n

if ENABLE_PREDICTOR:
    for dataset_name in DATASET_NAMES:
        print("Predictor:", dataset_name)
        model, history, k, n = train_or_load_predictor(dataset_name)
        PREDICTORS[dataset_name] = model
        PREDICTOR_HISTORIES[dataset_name] = history

In [ ]:
# Predictor training curves
if ENABLE_PREDICTOR:
    for dataset_name in DATASET_NAMES:
        history = PREDICTOR_HISTORIES[dataset_name]

        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(history["train_loss"])
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Training MSE")
        ax.set_yscale("log")
        ax.set_title(f"{dataset_name}: predictor training loss")
        ax.grid(True, alpha=0.3)
        plt.show()

In [ ]:
# ============================================================
# 16. One-shot predictor evaluation
# ============================================================

def predict_one_shot(dataset_name: str, count: Optional[int] = None):
    artifact = PCA_ARTIFACTS[dataset_name]
    bundle = load_dataset_bundle(dataset_name)
    model = PREDICTORS[dataset_name].eval()
    k = model.input_dim
    c, h, w = bundle.image_shape

    if count is None:
        count = min(10, len(bundle.test_images))
    z = artifact["test_coefficients"][:count, :k].to(DEVICE)
    target = bundle.test_images[:count].to(DEVICE)

    renderer = GaussianRenderer2D(
        h, w,
        chunk_size=64 if dataset_name == "STL10" else None
    ).to(DEVICE)

    with torch.no_grad():
        raw = model(z)
        reconstruction = renderer(raw)
        per_image_mse = ((reconstruction - target) ** 2).flatten(1).mean(dim=1)
        per_image_psnr = mse_to_psnr(per_image_mse)

    return {
        "target": target.cpu(),
        "reconstruction": reconstruction.cpu(),
        "raw_gaussians": raw.cpu(),
        "psnr": per_image_psnr.cpu(),
    }

ONE_SHOT_RESULTS = {}
if ENABLE_PREDICTOR:
    rows = []
    for dataset_name in DATASET_NAMES:
        result = predict_one_shot(dataset_name)
        ONE_SHOT_RESULTS[dataset_name] = result
        rows.append({
            "Dataset": dataset_name,
            "Average one-shot PSNR": float(result["psnr"].mean()),
            "Std. PSNR": float(result["psnr"].std()),
            "N": result["raw_gaussians"].shape[1],
            "K": PREDICTORS[dataset_name].input_dim,
        })

        shown = min(4, len(result["target"]))
        fig, axes = plt.subplots(shown, 2, figsize=(6, 3 * shown))
        if shown == 1:
            axes = np.array([axes])
        for i in range(shown):
            show_image(result["target"][i], "Target", axes[i, 0])
            show_image(
                result["reconstruction"][i],
                f"One-shot, {result['psnr'][i]:.2f} dB",
                axes[i, 1],
            )
        plt.suptitle(dataset_name)
        plt.tight_layout()
        plt.show()

    one_shot_table = pd.DataFrame(rows)
    display(one_shot_table)

### Part 4 discussion

The one-shot predictor produces meaningful reconstructions on every dataset without any per-image optimization.

Olivetti achieves the highest average one-shot PSNR at 21.75 dB. This supports the expectation that aligned faces are well suited to a low-dimensional eigenspace representation. CIFAR10 reaches 20.66 dB, FashionMNIST 19.26 dB, MNIST 17.23 dB, and STL10 15.65 dB.

The predictor captures coarse dataset-level structure. On MNIST it captures the approximate digit shape and stroke distribution. On FashionMNIST it captures the global clothing silhouette. On CIFAR10 it represents broad object and background colors but loses fine texture. On Olivetti it captures the global face layout and intensity structure. STL10 is the weakest case because 300 Gaussians and 256 PCA coefficients are still limited relative to the complexity and native resolution of the images.

The standard deviations, which range from approximately 1.7 to 2.2 dB, show that one-shot quality also varies across individual held-out images. Images close to the dominant training distribution are reconstructed more accurately, while unusual poses, backgrounds, or textures are harder.

The predictor is trained without Gaussian ground-truth labels. Its only supervision is the image-space MSE propagated through the differentiable renderer, so it learns Gaussians that are useful for reconstruction rather than reproducing any unique canonical Gaussian decomposition.

# Part 5 — Regular versus PCA initialization

Both methods use exactly the same:

- renderer;
- number of Gaussians;
- optimizer;
- learning rate;
- iteration budget;
- target images.

They differ only in initialization:

- **Regular splatting:** random Gaussian parameters.
- **PCA splatting:** Gaussian parameters predicted by the Part 4 model.

This controlled comparison isolates the effect of initialization.

In [ ]:
# ============================================================
# 17. Part 5 benchmark
# ============================================================

PART5_RESULTS = {}

def run_single_benchmark_case(
    dataset_name: str,
    image_index: int,
) -> Dict[str, Any]:
    bundle = load_dataset_bundle(dataset_name)
    artifact = PCA_ARTIFACTS[dataset_name]
    model = PREDICTORS[dataset_name].eval()

    target = bundle.test_images[image_index]
    _, h, w = target.shape
    n = model.num_gaussians
    k = model.input_dim

    renderer = GaussianRenderer2D(
        h, w,
        chunk_size=64 if dataset_name == "STL10" else None,
    ).to(DEVICE)

    z = artifact["test_coefficients"][image_index:image_index + 1, :k].to(DEVICE)
    with torch.no_grad():
        pca_initial = model(z)[0]

    random_initial = initialize_random_gaussians(
        n,
        seed=SEED + 1000 + image_index,
        device=DEVICE,
    )

    common_kwargs = dict(
        target=target,
        renderer=renderer,
        num_iterations=CFG["benchmark_iterations"],
        learning_rate=0.03,
        verbose=False,
    )

    regular = optimize_gaussians(
        initial_raw_gaussians=random_initial,
        **common_kwargs,
    )
    pca = optimize_gaussians(
        initial_raw_gaussians=pca_initial,
        **common_kwargs,
    )

    with torch.no_grad():
        initial_regular_img = renderer(random_initial).cpu()
        initial_pca_img = renderer(pca_initial).cpu()

    initial_regular_mse = F.mse_loss(initial_regular_img, target).item()
    initial_pca_mse = F.mse_loss(initial_pca_img, target).item()

    regular_final_psnr = regular.psnr_history[-1]
    pca_curve = np.array(pca.psnr_history)
    reached = np.where(pca_curve >= regular_final_psnr)[0]
    iterations_to_regular_quality = int(reached[0] + 1) if len(reached) else None
    speedup = (
        CFG["benchmark_iterations"] / iterations_to_regular_quality
        if iterations_to_regular_quality is not None
        else None
    )

    return {
        "dataset": dataset_name,
        "image_index": image_index,
        "target": target.cpu(),
        "regular": regular,
        "pca": pca,
        "initial_regular_psnr": float(-10 * np.log10(max(initial_regular_mse, 1e-12))),
        "initial_pca_psnr": float(-10 * np.log10(max(initial_pca_mse, 1e-12))),
        "iterations_to_regular_quality": iterations_to_regular_quality,
        "speedup": speedup,
    }

def run_part5_dataset(dataset_name: str) -> List[Dict[str, Any]]:
    count = min(CFG["benchmark_images"], len(load_dataset_bundle(dataset_name).test_images))
    payload = {
        "part": 5,
        "dataset": dataset_name,
        "count": count,
        "iterations": CFG["benchmark_iterations"],
        "seed": SEED,
        "run_mode": RUN_MODE,
        "version": 2,
    }
    path = RESULTS_DIR / "part5" / f"{dataset_name}_{config_hash(payload)}.pt"

    if USE_CACHE and path.exists() and not FORCE_RECOMPUTE:
        return load_torch(path)

    results = []
    for idx in range(count):
        print(f"Part 5: {dataset_name}, image {idx + 1}/{count}")
        results.append(run_single_benchmark_case(dataset_name, idx))
    save_torch(results, path)
    return results

if ENABLE_PART_5:
    for dataset_name in DATASET_NAMES:
        PART5_RESULTS[dataset_name] = run_part5_dataset(dataset_name)

In [ ]:
# Benchmark table
if ENABLE_PART_5:
    summary_rows = []

    for dataset_name, cases in PART5_RESULTS.items():
        summary_rows.append({
            "Dataset": dataset_name,
            "Images": len(cases),
            "Regular initial PSNR": np.mean([x["initial_regular_psnr"] for x in cases]),
            "PCA initial PSNR": np.mean([x["initial_pca_psnr"] for x in cases]),
            "Regular final PSNR": np.mean([x["regular"].psnr_history[-1] for x in cases]),
            "PCA final PSNR": np.mean([x["pca"].psnr_history[-1] for x in cases]),
            "Regular final loss": np.mean([x["regular"].loss_history[-1] for x in cases]),
            "PCA final loss": np.mean([x["pca"].loss_history[-1] for x in cases]),
            "Mean iterations to regular final quality": np.nanmean([
                np.nan if x["iterations_to_regular_quality"] is None
                else x["iterations_to_regular_quality"]
                for x in cases
            ]),
            "Mean speed-up": np.nanmean([
                np.nan if x["speedup"] is None else x["speedup"]
                for x in cases
            ]),
        })

    benchmark_table = pd.DataFrame(summary_rows)
    display(benchmark_table)

In [ ]:
# Mean PSNR against iteration for both methods
if ENABLE_PART_5:
    for dataset_name, cases in PART5_RESULTS.items():
        regular_curves = np.array([x["regular"].psnr_history for x in cases])
        pca_curves = np.array([x["pca"].psnr_history for x in cases])

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(regular_curves.mean(axis=0), label="Regular initialization")
        ax.plot(pca_curves.mean(axis=0), label="PCA initialization")
        ax.set_xlabel("Iteration")
        ax.set_ylabel("Mean PSNR (dB)")
        ax.set_title(f"{dataset_name}: convergence comparison")
        ax.grid(True, alpha=0.3)
        ax.legend()
        plt.show()

In [ ]:
# Required visual comparison:
# target, regular reconstruction, regular error,
# PCA reconstruction, PCA error.
if ENABLE_PART_5:
    for dataset_name, cases in PART5_RESULTS.items():
        rows = len(cases)
        fig, axes = plt.subplots(rows, 5, figsize=(15, 3 * rows))
        if rows == 1:
            axes = np.expand_dims(axes, axis=0)

        for row, case in enumerate(cases):
            target = case["target"]
            regular = case["regular"].final_image
            pca = case["pca"].final_image
            err_regular = (target - regular).abs().mean(dim=0, keepdim=True).repeat(3, 1, 1)
            err_pca = (target - pca).abs().mean(dim=0, keepdim=True).repeat(3, 1, 1)

            show_image(target, "Ground truth", axes[row, 0])
            show_image(
                regular,
                f"Regular\n{case['regular'].psnr_history[-1]:.2f} dB",
                axes[row, 1],
            )
            show_image(err_regular, "|GT - Regular|", axes[row, 2])
            show_image(
                pca,
                f"PCA\n{case['pca'].psnr_history[-1]:.2f} dB",
                axes[row, 3],
            )
            show_image(err_pca, "|GT - PCA|", axes[row, 4])

        plt.suptitle(f"{dataset_name}: regular vs PCA initialization")
        plt.tight_layout()
        plt.show()

## Domain-shift experiment

A predictor trained on one dataset is applied to an image from another distribution.  
Because the PCA coordinate systems have different dimensions and meanings, the foreign image is first resized to the source predictor's image shape and projected into the source PCA basis.

The expected result is poor initialization because the source eigenspace and predictor encode the structure of the training distribution, not arbitrary images.

In [ ]:
# ============================================================
# 18. Domain shift
# ============================================================

def resize_image_tensor(image: torch.Tensor, shape: Tuple[int, int, int]) -> torch.Tensor:
    c, h, w = shape
    image = image.unsqueeze(0)
    resized = F.interpolate(image, size=(h, w), mode="bilinear", align_corners=False)[0]
    if resized.shape[0] == 1 and c == 3:
        resized = resized.repeat(3, 1, 1)
    return resized

def domain_shift_experiment(
    source_dataset: str = "FashionMNIST",
    target_dataset: str = "MNIST",
    target_index: int = 0,
):
    source_artifact = PCA_ARTIFACTS[source_dataset]
    source_bundle = load_dataset_bundle(source_dataset)
    target_bundle = load_dataset_bundle(target_dataset)
    model = PREDICTORS[source_dataset].eval()

    target = target_bundle.test_images[target_index]
    target_resized = resize_image_tensor(target, source_bundle.image_shape)

    flat = target_resized.flatten()
    mean = source_artifact["mean"]
    components = source_artifact["components"][: model.input_dim]
    z = (flat - mean) @ components.T

    _, h, w = source_bundle.image_shape
    renderer = GaussianRenderer2D(h, w).to(DEVICE)

    with torch.no_grad():
        raw = model(z.unsqueeze(0).to(DEVICE))[0]
        reconstruction = renderer(raw).cpu()

    fig, axes = plt.subplots(1, 3, figsize=(10, 3))
    show_image(target, f"Original {target_dataset}", axes[0])
    show_image(target_resized, f"Resized to {source_dataset}", axes[1])
    show_image(reconstruction, f"{source_dataset} predictor output", axes[2])
    plt.tight_layout()
    plt.show()

    return reconstruction

if ENABLE_PREDICTOR and ENABLE_PART_5:
    DOMAIN_SHIFT_OUTPUT = domain_shift_experiment()

### Part 5 analysis

#### Initial quality

PCA initialization is clearly better before optimization on all five datasets.

The average initial PSNR improves from 8.45 to 17.55 dB on MNIST, from 8.87 to 19.39 dB on FashionMNIST, from 11.92 to 21.07 dB on CIFAR10, from 13.73 to 22.05 dB on Olivetti, and from 11.32 to 15.68 dB on STL10.

The largest initial gain is obtained on FashionMNIST, approximately 10.52 dB, followed by CIFAR10 and MNIST. STL10 receives the smallest gain, approximately 4.36 dB. This confirms that the PCA-to-Gaussian predictor learns a meaningful dataset-specific initialization.

#### Final quality

Despite its much better initial PSNR, PCA initialization does not produce the best final reconstruction under the current optimization settings.

Regular splatting reaches 35.33 dB on MNIST, 36.10 dB on FashionMNIST, 39.92 dB on CIFAR10, 36.46 dB on Olivetti, and 28.41 dB on STL10. PCA splatting reaches 22.64, 24.76, 37.73, 30.76, and 24.23 dB respectively.

The final deficit of PCA splatting is largest on MNIST and FashionMNIST, while CIFAR10 is the closest comparison, with a difference of approximately 2.18 dB.

Therefore, the experiment supports only part of the EigenGS-inspired hypothesis: PCA provides a substantially better immediate reconstruction, but in this implementation it usually converges to a worse final solution than random initialization.

#### Convergence speed

CIFAR10 is the only dataset for which the reported PCA runs reach the final quality of regular splatting within the iteration budget. The reported mean is 890.5 iterations, corresponding to a speed-up of approximately 1.13×.

For MNIST, FashionMNIST, Olivetti, and STL10, the benchmark reports no such crossing within 1000 iterations. Consequently, a speed-up factor cannot be reported for those datasets.

The CIFAR10 speed-up should also be interpreted cautiously because the reported mean is calculated only from cases that reached the target quality; cases that did not reach it are stored as missing values.

#### Cross-dataset interpretation

The initialization advantage is strongest for datasets with repeatable global structure. Olivetti obtains a strong initial PSNR because aligned faces are represented efficiently by PCA. FashionMNIST and MNIST also receive large initial gains because their objects are centered and have relatively consistent structure.

STL10 benefits least because its high-resolution natural images contain large variation in object position, scale, texture, and background. Its PCA model also did not reach 90% explained variance within 256 components.

However, the final optimization results show that a good image-space initialization does not automatically imply a good optimization basin. The predicted Gaussian parameters may contain broad scales, saturated activations, or overlapping primitives that initially approximate the image but are difficult to refine. Random Gaussians begin with worse quality but may remain more flexible during long optimization.

#### Error maps and failure cases

The early advantage of PCA initialization is consistent with its much higher initial PSNR. After the full optimization budget, however, the regular reconstructions generally have lower final MSE, as confirmed by the numerical table.

The visual error maps should therefore not be described as proving that PCA is always closer after optimization. Instead, they should be interpreted together with the PSNR table: PCA provides a strong starting point, but regular optimization usually overtakes it by the end of the run.

#### Overall answer

The controlled experiment shows that PCA/eigenspace information can be used successfully to predict an immediate 2D Gaussian representation. It improves the initial reconstruction on every dataset. Under the current model, parameterization, and optimization budget, however, this advantage usually does not translate into better final quality or substantial convergence speed-up. The result is therefore positive for one-shot initialization but mixed for continued optimization.

In [ ]:
# ============================================================
# 19. Compact runtime summary
# ============================================================

summary = {
    "run_mode": RUN_MODE,
    "device": str(DEVICE),
    "part2_enabled": ENABLE_PART_2,
    "part3_enabled": ENABLE_PART_3,
    "pca_enabled": ENABLE_PCA,
    "predictor_enabled": ENABLE_PREDICTOR,
    "part5_enabled": ENABLE_PART_5,
}

if ENABLE_PART_2 and PART2_RESULTS:
    summary["part2_total_runtime_seconds"] = float(sum(
        result.runtime_seconds
        for dataset_results in PART2_RESULTS.values()
        for result in dataset_results.values()
    ))

if ENABLE_PART_3 and PART3_RESULTS:
    summary["part3_total_runtime_seconds"] = float(sum(
        result.runtime_seconds for result in PART3_RESULTS.values()
    ))

display(pd.DataFrame([summary]))
print("Notebook pipeline completed.")

### Domain-shift discussion

The FashionMNIST predictor produces a poor representation of the MNIST digit because both the PCA basis and the Gaussian predictor are dataset-specific.

The MNIST image is projected into the FashionMNIST eigenspace, whose components describe clothing silhouettes rather than digit strokes. The resulting coefficients therefore do not have the same meaning as coefficients computed for an in-distribution FashionMNIST image. The MLP then maps these out-of-distribution coefficients to Gaussians using relationships learned only from clothing images.

This experiment demonstrates that the predictor does not learn a universal image-to-Gaussian mapping. It learns a mapping tied to the statistics, alignment, and appearance distribution of its training dataset.

This experiment highlights that the learned initialization is dataset-specific rather than universal, motivating the use of separate PCA spaces and predictors for different image domains.